# Job Listing Prioritization for Career Changers: EDA/SDA 1 (Full File, Field Comparison and Scope)

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

For a career changer, the project suggests the computer and mathematical roles their skills transfer to, hides the listings they can't get, and ranks the rest; the **retraining compass** is the foundation. **This notebook** cleans PredictLeads' full job-openings file, compares computer and mathematical roles with every other field, applies the filters we already know we want (the scope: **the roles TripleTen trains for, plus adjacent tech roles**), and explores the scope by role bucket:
- **Q3 (section 7):** how do computer and mathematical roles compare with every other field?
- **Q1 (9.1):** which factors should drive prioritization for a career changer?
- **Q2 (9.2):** what time information can prioritization rely on?

**The data:** PredictLeads' export of **September 30, 2026, the data's end date** (PredictLeads exports its whole database daily and calls each export a *snapshot*): every listing its crawler found from March 30 to September 30, with the first and last day it saw each one. **"Still open"** means last seen within 2 days before September 30. 13,415,388 listings; 13,017,464 English (or unlabeled) and US (or no country). Sections 2–5 show the raw structure on the first 200,000 documents; the analysis uses the full file.

**Hands off to EDA_SDA_2** (section 11): the scope table (TripleTen-role and adjacent tech listings, every field), their full descriptions, and two summaries of every field (occupation pay and timing, company activity).

**Conventions:**
- Every decision is recorded with its reasoning; the filters and why (section 8).
- Fix or flag, never drop, except data errors; reposts become a feature.
- Aggregates only: shares, medians and compact counts (13.0M, 412K); never rows. Clear outputs before committing.
- Each table or chart group ends with **What the data shows**: up to three bullets.

**The series** (each notebook hands its final table to the next):

| # | Notebook | Question | Hands off |
|:-:|:--|:--|:--|
| **1** | **EDA_SDA_1 (you are here)** | **What's in the full file, how does tech compare, and what's in the scope?** | **Scope table (every field) and its descriptions; occupation summary and company activity (every field)** |
| 2 | EDA_SDA_2 | Which scope listings do we keep and how do they behave? Additional cleaning, ghost listings, non-text checks by role bucket, blockers hidden, the compass's 2026 side | Scope table, non-text features |
| 3 | EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Scope table + text features |
| 4 | EDA_SDA_4 | Which features hold up? Statistics | Modeling-ready table |
| 5 | TimeSeries | When do listings appear and close? | Timing features |
| 6 | ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| 7 | ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |
| side | Profiles (frozen) | Who changed careers into or within tech or math, and how? (public resume dataset; three career-changer pools) | The compass's past side (path tables) and the modeling table for ML |

## 1. Setup

Libraries and the data loader, `src/pl_data.py`: the S3 connection, cached raw documents, resumable flattening and the schema writer. Sections 2–5 each call one of its functions.

In [ ]:
# 1a Libraries and the data loader (src/pl_data.py), plus the full-file rows table (src/full_rows.py)
import gc
import json
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))  # so `src` can be imported from notebooks/
from src.pl_data import (DATA_DIR, FLAT_DIR, JOBS_N, describe_structure, flatten_all, load_flat, load_raw, map_parts,
                         write_schemas)
pd.set_option("display.max_rows", 200)  # long summary tables (the feature dictionary) show in full
pd.set_option("display.max_colwidth", 140)
_marks = [time.time()]
def section_done(label):
    """Print how long a section took and the total so far, so a long run shows its progress."""
    now = time.time()
    print(f"[{label} done in {(now - _marks[-1]) / 60:.1f} min; {(now - _marks[0]) / 60:.1f} min in total]")
    _marks.append(now)

from src.eda_helpers import cell_format, column_heatmap, compact, show_shaded  # table formats and shading
from src.full_rows import (bit_counts, cat_contains, cat_map, has_label, load_rows, relabel, row_extras, save_final,
                           with_missing)

## 2. Load raw data

Both datasets are files in S3, part of a full export PredictLeads makes every day (it calls each day's export a *snapshot*); we use the one from September 30, 2026, stored as compressed JSON lines (`.jsonl.zst`): **companies** (~252 MB compressed) are loaded in full; **job openings** (~19 GB) are too big for 8 GB of RAM, so we read the first 200,000 documents. Each is cached as Parquet in `data/`, so later runs take seconds instead of about 10 minutes. Once every table is flattened (section 4), this step skips the raw documents and S3 entirely.

**Sections 2–5 use these 200,000 documents** to show the raw structure, flatten it and write the schemas. **The analysis from section 6 on uses the full file:** `src/stream_full.py` reads all 13.4 million listings from S3 line by line and saves 6.1e's cleaned row for every English (or unlabeled) US (or no-country) listing, 13,017,464 rows (`python -m src.stream_full --rows`).

In [ ]:
# 2a Load the raw documents ({} when every table is already flattened)
raw = load_raw()

## 3. Inspect structure

Each line of a file is a JSON:API document with two keys: `data` (the records) and `included` (related objects the records point to, such as the company behind a job opening). Column names and Python types only.

In [ ]:
# 3a Column names and the Python type of each value in the first row
describe_structure(raw)

## 4. Flatten

Turns the nested documents into ordinary tables, one row per record, with nested fields as dotted columns (for example `attributes.first_seen_at`): the companies and job openings, and each file's related objects with duplicates removed. Each table is built in parts of 20,000 records saved to `data/flat/` as they finish, so after a crash a rerun skips finished parts, and a finished table is never rebuilt.

In [ ]:
# 4a Flatten all four tables (skipped when already done)
flatten_all(raw)

## 5. Schema

Writes a `df.info()`-style summary of every flattened table (column names, non-null counts, dtypes) into `docs/data.md`, only when something changed.

In [ ]:
# 5a Write the schemas into docs/data.md
write_schemas()

## 6. Data cleaning

Cleaning comes before any analysis, because every timing factor depends on valid dates. **Data-quality fixes happen here** (impossible dates, duplicates, placeholders, mixed units, inconsistent labels). **Scope decisions wait for section 10** and are applied together in section 11, since they're judgment calls that need the full data as evidence. Reposts are kept as a feature, not removed. Each step shows summary tables only and logs the rows it touched; 6.7 has the full cleaning report.

### 6.0 Chart style

One style for every chart in the notebook: a light surface, recessive grid, thin marks, and a fixed color order (blue, orange, aqua, yellow) so a color always means the same thing within a chart. Charts show aggregates only.

In [ ]:
# 6.0a Chart colors and style shared by every chart below
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"  # categorical order: never reorder
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#c9c8c2", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "font.size": 10.5, "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

In [ ]:
# 6.0b Chart helpers
def bar_h(ax, s, title, xlabel, fmt="{:,.0f}", color=BLUE):
    """Horizontal bars, largest at the top, each value labelled at the bar end."""
    s = s.dropna().sort_values()
    ax.barh([str(i)[:42] for i in s.index], s.values, color=color, height=0.65)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=INK_2, fontsize=8.5)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.18)

def finish():
    plt.tight_layout()
    plt.show()

*No output: this cell only sets the chart style.*

### 6.1 Cleaned rows and text cleaning

The stream (`src/stream_full.py --rows`) ran 6.1e on every listing in the file and saved the cleaned rows as typed Parquet, without text. This section loads them.

**Decisions, and why:**
- **Descriptions are cleaned** (HTML removed, entities decoded, whitespace collapsed), so word counts measure words, not markup.
- **Descriptions become features:** `has_desc`, `has_html`, `desc_words`, `mentions_remote`, `urgency` (saved, but analyzed only in EDA_SDA_3's NLP), and `desc_hash` (the repost fingerprint for 6.4). *Why:* memory; the full text stays in the stream's scope parts for NLP.
- **Career-change flags:** `years_required` (most years asked, lower end of each range; blank if none; above 20 ignored), `degree_required`, `equivalent_experience`, `entry_wording`, `changer_wording`. *Why:* what a listing asks of a newcomer is known the moment it appears. They're keyword rules, so EDA_SDA_3 checks them for false positives.
- **Without a description, text flags are unknown (blank), not False:** "we can't tell" isn't "it doesn't say so". `mentions_remote` is still True when the title says remote. Comparisons (Q1f, Q1g) set listings with a flag against listings *known* not to have it.
- **The `url` becomes its source:** a hiring platform, a job board, or the company's own site. Full URLs aren't needed.
- **13 million rows in 8 GB of RAM:** only light columns are held in memory (dates, numbers, flags, repeated text as categories). Titles, descriptions and list fields are reduced part by part to fingerprints, counts and flags (`src/full_rows.py`), and the text and list columns are added back from disk when the final table is saved (11c).

**Every field in the original job-openings table, and where it's kept** (34 flattened columns)

The stream saved two sets: **every listing** (13.0M cleaned rows, this section, no long text) and **the scope listings** (about 1.0M computer and mathematical listings, every original field, including the full description). The scope table (section 11) carries both; the full descriptions sit next to it in `scope_descriptions.parquet`.

| Column | Description | Where it's kept |
|:--|:--|:--|
| `id` | Listing id; links every table and the descriptions file | Every listing |
| `type` | Record type, the same for every listing (kept for completeness) | Scope (`record_type`) |
| `attributes.title` | Title as posted, plus `title_norm` (lowercased, spaces collapsed) to match reposts | Every listing |
| `attributes.translated_title` | Title translated by PredictLeads; almost always empty. Only a flag is kept, since the scope is English only | Every listing, as `was_translated` |
| `attributes.normalized_title` | PredictLeads' standardized title (`job_title`), falling back to the posted title where it's missing (`job_title_fallback`) | Every listing |
| `attributes.description` | Full listing text. Too large for 13M listings (about 50 GB), so every listing keeps counts and flags (`has_desc`, `has_html`, `desc_words`, the text flags, the repost fingerprint `desc_hash`) | **Full text: scope only** (`scope_descriptions.parquet`); counts and flags: every listing |
| `attributes.url` | Link to the listing. Every listing keeps its source (`source_type`, `platform`); the scope keeps the link, which the app needs | Every listing (source); scope (`url`) |
| `attributes.first_seen_at` | When the crawler first found the listing (`first_seen`): freshness, `days_open`, `month` | Every listing |
| `attributes.last_seen_at` | When the crawler last saw it (`last_seen`): `live` and `days_open` | Every listing |
| `attributes.last_processed_at` | When PredictLeads last processed the listing (`last_processed`); a leakage audit only, since it likely moves with `last_seen` | Every listing |
| `attributes.contract_types` | Full time, part time, remote and so on (a list; labels unified) | Every listing |
| `attributes.categories` | PredictLeads category labels (a list) | Every listing |
| `attributes.onet_data.code` | O*NET-SOC occupation code (`onet_code`): role buckets and job growth join on it | Every listing |
| `attributes.onet_data.family` | O*NET occupation group (`occupation_group`): the field comparison (section 7) and the scope cut | Every listing |
| `attributes.onet_data.occupation_name` | O*NET occupation name (`occupation`) | Every listing |
| `attributes.posted_at` | The employer's posting date, where given (`posted`); implausible dates blanked (6.3) | Every listing |
| `attributes.recruiter_data.name` | Recruiter's name: personal data, never stored. An anonymous key per recruiter (`recruiter_key`, salted fingerprint of name and company) lets us study recruiters without knowing who they are | Scope, as `recruiter_key`; every listing, as the flag `has_recruiter` |
| `attributes.recruiter_data.title` | The recruiter's job title, e.g. technical recruiter (`recruiter_title`) | Scope |
| `attributes.recruiter_data.contact` | Recruiter's contact details: personal data | **Not kept** (only the `has_recruiter` flag) |
| `attributes.salary` | Pay as written (`salary_text`); parsed into hourly pay where `salary_data` has no numbers (8d) | Scope |
| `attributes.salary_data.salary_low` | Low end of the pay range, in the posted currency (`salary_low`) | Every listing |
| `attributes.salary_data.salary_high` | High end of the pay range, in the posted currency (`salary_high`) | Every listing |
| `attributes.salary_data.salary_currency` | Currency of the posted pay (`salary_currency`) | Every listing |
| `attributes.salary_data.salary_low_usd` | Low end converted to USD (`salary_low_usd`) | Every listing |
| `attributes.salary_data.salary_high_usd` | High end converted to USD (`salary_high_usd`) | Every listing |
| `attributes.salary_data.salary_time_unit` | Hour, day, week, month or year (`salary_unit`): converts pay to `salary_hourly_usd` (6.5) | Every listing |
| `attributes.seniority` | Seniority level (`seniority`) | Every listing |
| `attributes.status` | PredictLeads status, only "closed" or empty (`status`); not used as a live flag: `live` comes from `last_seen` | Every listing |
| `attributes.language` | Listing language (`language`): the English-only filter | Every listing |
| `attributes.location` | Location as written (`location_text`), e.g. a city or "Remote - US"; for the app's location filter | Scope |
| `attributes.location_data` | Structured locations (a list of places). Every listing keeps `country` (first location) and `n_locations`; EDA_SDA_2 reads the state from the scope's records | Every listing (country, count); scope (full records, in the stream's scope parts) |
| `attributes.tags` | PredictLeads tags (a list; about half empty) | Every listing |
| `relationships.company.data.id` | The hiring company's id (`company_id`): company volume, links to the companies table | Every listing |
| `relationships.company.data.type` | Type of the linked company record, the same for every listing (kept for completeness) | Scope (`company_record_type`) |

The file's related objects (`job_openings_included`: each company's domain, name and stock ticker) come from the companies table through `company_id` (EDA_SDA_2).

In [ ]:
# 6.1a Paths and the columns we keep
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

from src.listing_rows import COLS  # flattened column -> short name; shared with the full-file stream (src/stream_full.py)

In [ ]:
# 6.1b Patterns used to clean and flag description text. They live in src/listing_rows.py so this notebook and the
# full-file stream use one copy; EDA_SDA_3 validates them. Text: TAG_RE (HTML tags), URGENT_RE; career-change
# accessibility: YEARS_RE (with MAX_YEARS = 20, since "50 years of experience" is the company's history), ENTRY_RE,
# CHANGER_RE, DEGREE_RE, EQUIV_RE.
from src.listing_rows import (CHANGER_RE, DEGREE_RE, ENTRY_RE, EQUIV_RE, MAX_YEARS, TAG_RE, URGENT_RE, YEARS_RE,
                              years_required)

In [ ]:
# 6.1c Posting source from the url domain: hiring platforms (applicant tracking systems) and job boards, matched against
# the url's domain (ATS and BOARDS in src/listing_rows.py); everything else is "company site / other".
from src.listing_rows import ATS, BOARDS, source_of

In [ ]:
# 6.1d Text and location helpers: clean_text strips HTML and decodes entities; text_flag is True / False from the
# description and unknown (NA) without one; first_country takes the first of a listing's locations.
from src.listing_rows import clean_text, first_country, text_flag

In [ ]:
# 6.1e Turn one flattened part into lean rows: listing columns, text flags (unknown without a description), description
# length, the repost fingerprint (desc_hash) and the posting source. Shared with the full-file stream.
from src.listing_rows import lean_part

In [ ]:
# 6.1f Load the cleaned rows of the full file (src/stream_full.py --rows: 6.1e on every English US listing).
# Light columns only, so 13 million rows fit in 8 GB: repeated text as categories, flags as true / false / unknown.
# row_extras reads titles, ids and list fields part by part and keeps fingerprints, counts and bit flags instead.
LOAD_COLS = ["company_id", "first_seen", "last_seen", "posted", "status", "seniority", "language", "onet_code",
             "occupation_group", "occupation", "salary_low_usd", "salary_high_usd", "salary_unit", "has_desc", "has_html",
             "desc_words", "mentions_remote", "urgency", "years_required", "entry_wording", "changer_wording",
             "degree_required", "equivalent_experience", "n_locations", "country", "source_type", "platform"]
start = time.time()
stream = json.loads((FLAT_DIR / "stream_rows_progress.json").read_text())  # counts from the stream (no rows)
if not stream.get("finished"):
    raise RuntimeError("The rows stream hasn't finished: run `python -m src.stream_full --rows` first.")
jobs = load_rows(LOAD_COLS)
extras, labels = row_extras()  # same order as jobs: the index is each listing's row position in the rows table
jobs = pd.concat([jobs, extras], axis=1)
del extras
gc.collect()
print(f"loaded {len(jobs):,} cleaned rows × {jobs.shape[1]} columns in {time.time() - start:.0f}s, "
      f"{jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

companies = load_flat("companies", ["id", "attributes.location_data"])
companies["country"] = companies.pop("attributes.location_data").map(first_country)  # the company's first location (8b)
companies = companies.drop_duplicates("id")

In [ ]:
# 6.1g Helpers for the cleaning report and summary tables
from IPython.display import display

report = []  # cleaning audit trail: one row per action, shown in 6.7
def log(step, action, n):
    report.append({"step": step, "action": action, "rows": int(n)})

def show(title, df, by="columns"):
    """Print a heading, then display a summary table. Aggregates only, never rows.
    Numbers are formatted from the column name (or the row name, by="rows", when rows are measures): shares as
    percentages, days and years with one decimal, pay in dollars, counts compact (13.0M, 412K). describe() tables keep
    their count row as a count. Wide group comparisons use show_shaded instead."""
    df = df.to_frame() if isinstance(df, pd.Series) else df.copy()
    describe = {"count", "mean", "50%"} <= set(map(str, df.index))
    numeric = lambda s: pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)
    if by == "rows":
        out = df.astype(object)
        special = [c for c in df.columns if any(k in str(c) for k in ("rank", "spread"))]
        for r in df.index:
            f = cell_format(r, pd.to_numeric(df.loc[r].drop(special), errors="coerce"))
            for c in df.columns:
                v = df.at[r, c]
                if isinstance(v, (int, float, np.number)) and not isinstance(v, bool):
                    out.at[r, c] = cell_format(c, df[c])(v) if any(k in str(c) for k in ("rank", "spread")) else f(v)
        df = out
    else:
        for c in df.columns:
            if numeric(df[c]):
                f = cell_format(c, df[c].drop(index="count", errors="ignore") if describe else df[c])
                df[c] = [compact(v) if describe and str(i) == "count" else f(v) for i, v in df[c].items()]
    print(f"\n{title}")
    display(df)

def distinct(s):
    try:
        return s.nunique()
    except TypeError:  # list-valued columns
        return np.nan

In [ ]:
# 6.1h Summary tables
log("6.1", "rows in the cleaned rows table (every English or unlabeled, US or no-country listing)", len(jobs))

show(f"Cleaned rows: {compact(len(jobs))} rows × {jobs.shape[1]} columns in memory", jobs.notna().mean().to_frame("share filled"))
show("From the full file to the cleaned rows (the stream's filter, as decided in section 8)", pd.DataFrame({"listings": {
    "listings in the file": stream["records"],
    "English or unlabeled, and US or no country (kept)": stream["rows"],
    "other languages or countries (not kept)": stream["records"] - stream["rows"],
}}).assign(share=lambda d: (d.listings / stream["records"]).round(3)))

show("Text cleaning results (text flags: share of listings with a description)", pd.DataFrame({"share": {
    "has a description": jobs.has_desc.mean(),
    "description contained HTML (now stripped)": jobs.has_html.mean(),
    "mentions remote": jobs.mentions_remote.mean(),
    "states years of experience required": jobs.years_required.notna().mean(),
    "entry-level or training wording": jobs.entry_wording.mean(),
    "career-changer wording": jobs.changer_wording.mean(),
    "degree required": jobs.degree_required.mean(),
    "or equivalent experience": jobs.equivalent_experience.mean(),
}}).round(3))

show("Years of experience required (listings that state it)",
     jobs.years_required.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("years"))

show("Description length in words (postings with a description)",
     jobs.desc_words[jobs.has_desc].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(0).to_frame("words"))

show("Posting source (from the url domain)", pd.DataFrame({
    "postings": jobs.source_type.value_counts(),
    "share": jobs.source_type.value_counts(normalize=True).round(3),
}))

In [ ]:
# 6.1i Charts: description length and where listings are hosted
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.4, 1]})
ax1.hist(jobs.desc_words[jobs.has_desc].clip(upper=2000), bins=50, color=BLUE)
ax1.axvline(400, color=INK_2, ls="--", lw=1)
ax1.text(420, ax1.get_ylim()[1] * 0.92, "~400 words: where small\nembedding models cut off", color=INK_2, fontsize=8.5, va="top")
ax1.set(title="Description length", xlabel="words per description (capped at 2,000)", ylabel="listings")
bar_h(ax2, 100 * jobs.source_type.value_counts(normalize=True), "Where listings are hosted", "% of listings", "{:.1f}%")
finish()

**What the data shows**
- **13.0M English US listings** from the full file of 13.4M (3.0% removed for another language or country). 83% have a description, a median **608 words**, so later embedding work needs chunking.
- **Among listings with a description:** 13% mention remote work and 27% state years of experience (median 3; 90% ask for 8 or fewer). Only 10% use entry-level or training wording, 3.7% require a degree and **0.4% invite career changers**.
- **60% are on company sites or other domains,** 29% on hiring platforms and 11% on job boards.

### 6.2 Types and standard categories

**Decisions, and why:**
- **Dates become UTC timestamps,** so they can be compared. Text that won't parse becomes missing, and is counted: guessing would invent data.
- **Category labels are trimmed and lowercased,** and `contract_types` spellings are unified ("full_time" = "full time"), so one value counts once.
- **`title_norm`** (lowercased, whitespace collapsed) is used to match reposts: built part by part into the repost fingerprint (6.1f) and saved with the final table (11c).
- **Dates were parsed in the stream** (text that wouldn't parse became missing there).

In [ ]:
# 6.2a Dates: parsed to UTC timestamps in the stream (text that wouldn't parse became missing there)
date_rows = {col: {"has a date": jobs[col].notna().sum(), "missing": jobs[col].isna().sum(),
                   "earliest": jobs[col].min(), "latest": jobs[col].max()} for col in ["first_seen", "last_seen", "posted"]}
show("Dates", pd.DataFrame(date_rows).T)

In [ ]:
# 6.2b Standardize category labels (on each column's distinct labels, not row by row)
cat_rows, label_tables = {}, {}
for col in ["status", "seniority", "language", "salary_unit"]:
    before = jobs[col].nunique()
    jobs[col] = relabel(jobs[col], lambda c: str(c).strip().lower())
    log("6.2", f"{col}: distinct labels merged by trimming and lowercasing", before - jobs[col].nunique())
    cat_rows[col] = {"labels before": before, "labels after": jobs[col].nunique(),
                     "merged": before - jobs[col].nunique(), "missing %": round(100 * jobs[col].isna().mean(), 1)}
    counts = with_missing(jobs[col]).value_counts()
    label_tables[col] = counts[counts > 0].head(8)
show("Category labels: trimmed and lowercased", pd.DataFrame(cat_rows).T)
for col, counts in label_tables.items():
    show(f"Top {col} labels after cleaning", pd.DataFrame({"postings": counts, "share": (counts / len(jobs)).round(3)}))

In [ ]:
# 6.2c Normalized titles and unified contract_types labels: both built part by part in 6.1f (row_extras).
# The normalized title (lowercased, whitespace collapsed) goes into the repost fingerprint (6.4) and is saved as title_norm
# (11c). contract_types spellings of the same label ("full_time" and "full time") are unified, kept as bit flags here.
before, after = labels["raw_contract_labels"], len(labels["contract_types"])
log("6.2", "contract_types: distinct labels merged (underscores, case, spacing)", before - after)
show("contract_types labels", pd.DataFrame({"labels": {"before": before, "after": after, "merged": before - after}}))
show("Titles", pd.DataFrame({"share of listings": {"with a title": jobs.has_title.mean(), "without a title": (~jobs.has_title).mean()}}).round(4))

**What the data shows**
- **The dates are complete:** every listing has a first and last sighting between March 30 and September 30. The employer's posting date exists for 44%, with placeholders (1969) and future dates handled in 6.3.
- **Labels were already clean** at full scale: trimming and lowercasing merged nothing. Contract types are the exception: 166 labels, only 5 merged by spelling, and many overlapping ("intern" and "internship", "m/f"), which EDA_SDA_2 sorts out.
- **`status` is "closed" (65%) or empty, never "open".** Seniority: mid/senior 53%, junior 30%, manager 12%.

### 6.3 Invalid dates

**Decisions, and why:**
- **Missing `first_seen` or `last_seen`, `last_seen` before `first_seen`, or dates after September 30 (the data's end date): removed.** These are data errors, and such a listing can't be placed in time.
- **`posted` more than a day after `first_seen`, or before 2000: only `posted` is blanked.** The crawler can't find a listing before it exists, and 1969-12-30 is a placeholder; the listing itself stays.

In [ ]:
# 6.3a Remove impossible timelines
DATA_END = pd.Timestamp("2026-09-30", tz="UTC")
END_OF_DATA = DATA_END + pd.Timedelta(days=1)
rows_before = len(jobs)

rules = {
    "missing first_seen or last_seen": jobs.first_seen.isna() | jobs.last_seen.isna(),
    "last_seen before first_seen": jobs.last_seen < jobs.first_seen,
    "dates after September 30 (the data's end date)": (jobs.first_seen > END_OF_DATA) | (jobs.last_seen > END_OF_DATA),
}
rule_rows, bad = {}, pd.Series(False, index=jobs.index)
for name, mask in rules.items():
    new = (mask & ~bad).sum()  # count each row once, under the first rule it fails
    log("6.3", f"removed, {name}", new)
    rule_rows[name] = {"rows": new, "share of rows": round(new / rows_before, 4), "action": "removed"}
    bad |= mask
jobs = jobs[~bad]
del rules
gc.collect()
missing_dates = (jobs.first_seen.isna() | jobs.last_seen.isna()).mean() if len(jobs) else 0.0
if rule_rows["missing first_seen or last_seen"]["share of rows"] > 0.01:
    print(f"WARNING: {rule_rows['missing first_seen or last_seen']['share of rows']:.1%} of listings have no first_seen or "
          "last_seen. On the 200,000-row sample this was close to 0%, so the stream's date parsing may have failed on some "
          "date formats: tell Claude before using the results.")

In [ ]:
# 6.3b Blank implausible posted dates
bad_posted = (jobs.posted > jobs.first_seen + pd.Timedelta(days=1)) | (jobs.posted < pd.Timestamp("2000-01-01", tz="UTC"))
jobs.loc[bad_posted, "posted"] = pd.NaT
log("6.3", "posted blanked (after first_seen or before 2000)", bad_posted.sum())
rule_rows["posted after first_seen or before 2000"] = {
    "rows": bad_posted.sum(), "share of rows": round(bad_posted.sum() / rows_before, 4), "action": "posted blanked"}

In [ ]:
# 6.3c Summary tables
show("Date validity rules", pd.DataFrame(rule_rows).T)
show("Rows before and after", pd.DataFrame({"rows": {"before": rows_before, "removed": rows_before - len(jobs),
                                                     "after": len(jobs)}}).assign(share=lambda d: (d.rows / rows_before).round(4)))
days_open = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
show("Days between first_seen and last_seen (after cleaning)",
     days_open.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))

In [ ]:
# 6.3d Chart: days between first seen and last seen
still_open = (DATA_END - jobs.last_seen).dt.days <= 2
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist([days_open[~still_open], days_open[still_open]], bins=60, stacked=True, color=[BLUE, ORANGE],
        label=["closed by September 30", "still open on September 30"])
ax.set(title="Days between first seen and last seen", xlabel="days", ylabel="listings")
ax.legend()
finish()

**What the data shows**
- **No impossible timelines:** no listing lacks a date, ends before it starts, or runs past September 30, so nothing was removed (the date-loss warning didn't trigger).
- **349K employer posting dates (2.7%) were blanked:** placeholders before 2000, or dates after the crawler first saw the listing.
- **Listings are seen for a median 16.7 days;** a quarter for 40+ days, and 10% for more than 78.

### 6.4 Duplicates and reposts

**Decisions, and why:**
- **The same record `id` twice:** one copy kept (a data error).
- **Reposts** (same company, normalized title and description fingerprint, new `id`; matched on `repost_fp`, one 64-bit fingerprint of the three) **are kept and flagged** with `is_repost`; the earliest copy is the original. Features use **earlier copies only**, so they're known when a listing appears: `times_listed`, `days_since_first_listed`, `days_since_repost_last_listed`. Whether a listing *will* be reposted is future information and isn't used. Listings without a description can't be matched.
- **The EDA tables count each job once** (originals, `jo` in 6.7): a counting view, not a filter.

In [ ]:
# 6.4a Drop exact duplicate ids; flag reposts of the same job
# Both use 64-bit fingerprints built in 6.1f (id_fp; repost_fp = company, normalized title and description fingerprint):
# one number per listing instead of long text keys. Two different keys sharing a fingerprint is about a 1-in-200,000
# chance across all 13 million listings.
rows_before = len(jobs)
dup_id = jobs.id_fp.duplicated()
log("6.4", "removed, duplicate id", dup_id.sum())
jobs = jobs[~dup_id].drop(columns="id_fp")

jobs = jobs.sort_values("first_seen")
key = ["repost_fp"]
has_key = jobs.repost_fp.notna()
jobs["is_repost"] = has_key & jobs.duplicated(key)
log("6.4", "flagged as repost (kept, excluded from the EDA tables)", jobs.is_repost.sum())
log("6.4", "postings without a description, can't be checked for reposts", (~has_key).sum())

# Repost features: built only from earlier copies of the same job, so they're known when a listing appears
same_job = jobs[has_key].groupby(key)
jobs["times_listed"] = np.float32(np.nan)  # unknown without a description
jobs.loc[has_key, "times_listed"] = (same_job.cumcount() + 1).astype("float32")  # jobs are sorted by first_seen
jobs["days_since_first_listed"] = np.float32(np.nan)  # since the job's original listing
jobs.loc[has_key, "days_since_first_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.transform("min")).dt.total_seconds().div(86400).astype("float32")
jobs["days_since_repost_last_listed"] = np.float32(np.nan)  # since the job's previous listing; 0 for the first listing
jobs.loc[has_key, "days_since_repost_last_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.shift()).dt.total_seconds().fillna(0).div(86400).astype("float32")
del same_job
_ = gc.collect()

In [ ]:
# 6.4b Summary tables
show("Duplicates and reposts", pd.DataFrame({"rows": {
    "before": rows_before,
    "exact duplicate ids (removed)": dup_id.sum(),
    "after": len(jobs),
    "reposts (flagged, kept)": jobs.is_repost.sum(),
    "originals": (~jobs.is_repost).sum(),
    "no description, can't be checked": (~has_key).sum(),
}}).assign(share=lambda d: (d.rows / rows_before).round(4)))
group_sizes = jobs[has_key].groupby(key).size()
show("How many times the same job is listed (original + reposts)",
     group_sizes.value_counts().sort_index().head(10).rename_axis("listings per job").to_frame("jobs"))
show("Repost features: times this job has been listed so far",
     jobs.times_listed.fillna(0).astype(int).clip(upper=5).value_counts().sort_index()
     .rename(index={0: "unknown (no description)", 5: "5+"}).rename_axis("times listed").to_frame("listings"))
show("Repost rate by source type", jobs.groupby("source_type", observed=True).is_repost.mean().round(3).to_frame("repost rate"))

In [ ]:
# 6.4c Charts: repost rate and listings per job
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
bar_h(ax1, 100 * jobs.groupby("source_type", observed=True).is_repost.mean(), "Repost rate by source type", "% of listings that are reposts", "{:.1f}%")
sizes = group_sizes.clip(upper=10).value_counts().sort_index()
ax2.bar([str(int(k)) if k < 10 else "10+" for k in sizes.index], sizes.values, color=BLUE, width=0.65)
ax2.set_yscale("log")
ax2.set(title="Listings per job (log scale)", xlabel="times the same job is listed", ylabel="jobs")
ax2.grid(axis="x", visible=False)
finish()

**What the data shows**
- **No duplicate ids.** 1.4M listings (10%) repeat an earlier listing's company, title and description; they're flagged as reposts and kept. 17% can't be checked, having no description.
- **Reposts come from job boards:** 32% of job-board listings are reposts, vs. 9% on company sites and 6% on hiring platforms, so job boards mostly re-list jobs posted elsewhere.
- **Most jobs are listed once** (8.9M), with a long tail: 612K listings are at least the fifth listing of their job.

### 6.5 Salary to one hourly rate

**Decisions, and why:**
- **The midpoint of the low and high values, converted to an hourly rate** (a year is 2,080 hours, 260 workdays, 52 weeks or 12 months), so hourly and salaried pay compare. Many career changers come from hourly jobs, so hourly reads naturally. Hourly listings keep their posted rate; converting salaried pay assumes full-time work.
- **Reversed ranges** (low above high) are swapped, and **clearly wrong units** are re-read: "$25 a year" is hourly pay, "$85,000 an hour" is yearly pay. Each is flagged `pay_suspect`.
- **A missing or unknown unit, or a result outside $4.81–$480.77 an hour** ($10k–$1M a year): only the salary is blanked, and counted.

In [ ]:
# 6.5a Salary time units
PER_YEAR = {"hour": 2080, "day": 260, "week": 52, "month": 12, "year": 1}

def unit_factor(u):
    if pd.isna(u):
        return np.nan
    return next((f for k, f in PER_YEAR.items() if k in u), np.nan)

mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
factor = cat_map(jobs.salary_unit, unit_factor)
unit_counts = with_missing(jobs.loc[mid.notna(), "salary_unit"]).value_counts()
unit_counts = unit_counts[unit_counts > 0]
show("Salary units (postings with a salary)", pd.DataFrame({
    "postings": unit_counts,
    "× per year": unit_counts.index.map(lambda u: unit_factor(u) if u != "(missing)" else np.nan),
}).fillna({"× per year": "not recognized"}))

In [ ]:
# 6.5b Fix reversed ranges and wrong units, convert to one hourly rate, blank outliers
HOURS_PER_YEAR = PER_YEAR["hour"]
reversed_range = jobs.salary_low_usd > jobs.salary_high_usd
jobs.loc[reversed_range, ["salary_low_usd", "salary_high_usd"]] = (
    jobs.loc[reversed_range, ["salary_high_usd", "salary_low_usd"]].to_numpy())
log("6.5", "salary range reversed (low above high): swapped", reversed_range.sum())

mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
yearly_but_hourly = cat_contains(jobs.salary_unit, "year") & mid.between(7, 200)              # "$25 a year" is hourly pay
hourly_but_yearly = cat_contains(jobs.salary_unit, "hour") & mid.between(10_000, 1_000_000)   # "$85,000 an hour" is yearly pay
factor = cat_map(jobs.salary_unit, unit_factor).mask(yearly_but_hourly, PER_YEAR["hour"]).mask(hourly_but_yearly, PER_YEAR["year"])
log("6.5", "labelled yearly but clearly hourly ($7-$200): re-read as hourly", yearly_but_hourly.sum())
log("6.5", "labelled hourly but clearly yearly ($10k-$1M): re-read as yearly", hourly_but_yearly.sum())
log("6.5", "salary present but unit missing or not recognized", (mid.notna() & factor.isna()).sum())

hourly = mid * factor / HOURS_PER_YEAR
outlier = ~hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR) & hourly.notna()
jobs["salary_hourly_usd"] = hourly.mask(outlier).astype("float32")
jobs["pay_suspect"] = reversed_range | yearly_but_hourly | hourly_but_yearly
log("6.5", "hourly pay outside $4.81-$480.77 ($10k-$1M a year), blanked", outlier.sum())

In [ ]:
# 6.5c Summary tables
show("Salary cleaning", pd.DataFrame({"postings": {
    "salary present": mid.notna().sum(),
    "range reversed (swapped)": reversed_range.sum(),
    "labelled yearly, re-read as hourly": yearly_but_hourly.sum(),
    "labelled hourly, re-read as yearly": hourly_but_yearly.sum(),
    "unit missing or not recognized": (mid.notna() & factor.isna()).sum(),
    "converted to hourly": hourly.notna().sum(),
    "outside $4.81–$480.77 an hour (blanked)": outlier.sum(),
    "usable hourly pay": jobs.salary_hourly_usd.notna().sum(),
}}).assign(**{"share of listings": lambda d: (d.postings / len(jobs)).round(4)}))
q = [0.1, 0.25, 0.5, 0.75, 0.9]
show("Hourly pay (USD) quantiles, before and after removing outliers", pd.DataFrame({
    "pay ($ an hour), before": hourly.quantile(q), "pay ($ an hour), after": jobs.salary_hourly_usd.quantile(q)}))

In [ ]:
# 6.5d Charts: salary units and hourly pay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.4]})
bar_h(ax1, unit_counts, "Salary time units", "listings with a salary")
pay = jobs.salary_hourly_usd.dropna().clip(upper=150)
ax2.hist(pay, bins=60, color=BLUE)
ax2.axvline(pay.median(), color=INK_2, ls="--", lw=1)
ax2.text(pay.median(), ax2.get_ylim()[1] * 0.95, f"  median ${pay.median():.2f}/hr", color=INK_2, fontsize=8.5, va="top")
ax2.set(title="Hourly pay (midpoint, full-time equivalent, capped at $150)", xlabel="USD per hour", ylabel="listings")
finish()

**What the data shows**
- **43% disclose pay** (5.6M listings), all with a recognized time unit: hourly for 24% of all listings, yearly for 18%.
- **At full scale nothing needed fixing:** no reversed ranges or mislabelled units; 984 values outside $10K–$1M a year were blanked.
- **Median $29.08 an hour across every field,** with the middle half between $20 and $50.

### 6.6 Missing values

The share of postings missing each field we plan to use. It shows what each analysis can rely on. For example, the crawler-vs-employer check in 7.1 only uses postings that have `posted`.

In [ ]:
# 6.6 Missing-value summary for the fields the analysis uses
fields = ["first_seen", "last_seen", "posted", "status", "seniority", "language",
          "occupation_group", "occupation", "salary_hourly_usd", "country", "platform"]
missing = pd.DataFrame({"missing %": 100 * jobs[fields].isna().mean(), "available rows": jobs[fields].notna().sum()})
missing.loc["title"] = [100 * (~jobs.has_title).mean(), jobs.has_title.sum()]
missing.loc["description"] = [100 * (~jobs.has_desc).mean(), jobs.has_desc.sum()]
show("Missing values (after cleaning)", missing.round(1).sort_values("missing %", ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
bar_h(ax, missing["missing %"], "Missing values by field", "% of listings missing", "{:.1f}%")
finish()

**What the data shows**
- **The timeline, title, seniority and occupation are complete;** the description is missing for 17%, the country for 8.0% and the language for 4.8%.
- **The sparse fields are pay (57% missing), the employer's posting date (58%) and the named platform (60%,** since company sites aren't named). Analyses that use them say so.

### 6.7 Cleaning report and derived timing columns

**Decisions, and why:**
- **`live`: last seen within 2 days of September 30 (the data's end date), so still open then,** since the crawler doesn't revisit every listing daily. A live listing's lifespan is *censored*, which Q1b handles.
- **`days_open`** (first seen to last seen) and **`month`** (month first seen, the time horizon).
- **`jo`: originals only, for the EDA tables;** every cleaned listing, reposts flagged, is saved to disk and starts the scope table in section 8 (memory: one copy in RAM at a time).
- **Crawl artifacts, flagged and kept (6.8):** `batch_first_seen` (a company's listings first seen in one batch, when the company was added to the crawl) and `crawl_dropout` (a company's listings that vanished on one day, when the crawler likely lost the site). Same definitions as the hackathon app's signals; timing results in section 9 are shown with and without them.
- **`company_earlier`:** the company's listings in any field first seen before this one, counted before the cut, so company volume reflects all of a company's hiring.
- **Career-change flags:** `entry_seniority`, `low_experience_ask` (2 years or fewer) and `career_open` (either of those, or entry-level/training or career-changer wording): a deliberately broad first pass for Q1g and Q3. "Or equivalent experience" is left out, since it usually means experience in the same field.

In [ ]:
# 6.7a Derived timing columns and the analysis table (originals only)
LIVE_DAYS = 2
jobs["live"] = (DATA_END - jobs.last_seen).dt.days <= LIVE_DAYS
jobs["days_open"] = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
jobs["month"] = jobs.first_seen.dt.tz_localize(None).dt.to_period("M")
jobs["occupation_group"] = with_missing(jobs.occupation_group)

# Career-change accessibility flags (see the decisions above)
ENTRY_LEVELS = {"junior"}  # checked in Q3b: junior is the only entry-level label (others: mid_senior, manager, head, director, vice_president, president, c_level, partner, founder, not_set)
jobs["entry_seniority"] = jobs.seniority.isin(ENTRY_LEVELS)
jobs["low_experience_ask"] = jobs.years_required <= 2  # False where no years are stated
jobs["career_open"] = jobs.entry_seniority | jobs.low_experience_ask | jobs.entry_wording | jobs.changer_wording

# Company history in any field, counted before the cut (company volume is a confounder for every later notebook):
# each listing's count of its company's listings first seen before it (same-day listings count together; reposts included)
jobs["company_earlier"] = (jobs.groupby("company_id", observed=True).first_seen.rank(method="min") - 1).astype("float32")

# Crawl artifacts (flagged, kept; definitions shared with the hackathon app's signals):
# - batch_first_seen: the company has 5+ listings and 60%+ of them were first seen on one day (the company was added
#   to the crawl, or its site rebuilt), and this listing is one of them. The data's first two days are left out: every
#   listing already up then is first seen there (they're flagged partial_month instead).
# - crawl_dropout: 5+ of the company's listings disappeared (not live) and 60%+ of those were last seen on one day
#   (the crawler likely lost the site), and this listing is one of them: its days_open may end early.
BATCH_MIN, BATCH_SHARE = 5, 0.6
def one_day_batch(company_codes, day, eligible):
    """True for listings on the day holding BATCH_SHARE+ of their company's eligible listings (BATCH_MIN+ of them)."""
    d = pd.DataFrame({"co": company_codes[eligible], "day": day[eligible]})
    d = d[d.co >= 0]
    per_day = d.groupby(["co", "day"]).size()
    per_co = per_day.groupby(level="co").sum()
    top = per_day.sort_values(ascending=False).groupby(level="co").head(1)  # the busiest day per company
    top = top[(per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_MIN)
              & (top.to_numpy() / per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_SHARE)]
    hit = pd.MultiIndex.from_arrays([company_codes, day]).isin(top.index)
    return pd.Series(hit & eligible, index=jobs.index)

codes = jobs.company_id.cat.codes.to_numpy()
first_day = (jobs.first_seen.astype("int64") // 86_400_000_000_000).to_numpy()
last_day = (jobs.last_seen.astype("int64") // 86_400_000_000_000).to_numpy()
jobs["batch_first_seen"] = one_day_batch(codes, first_day, (jobs.first_seen >= pd.Timestamp("2026-04-01", tz="UTC")).to_numpy())
jobs["crawl_dropout"] = one_day_batch(codes, last_day, (~jobs.live).to_numpy())
artifact_stats = {
    "listings first seen in a company's one-day batch": jobs.batch_first_seen.mean(),
    "companies with such a batch (of companies with 5+ listings)": (
        jobs.loc[jobs.batch_first_seen, "company_id"].nunique() / max((jobs.company_id.value_counts() >= BATCH_MIN).sum(), 1)),
    "listings that disappeared in a company's one-day dropout": jobs.crawl_dropout.mean(),
    "companies with such a dropout (of companies with 5+ listings)": (
        jobs.loc[jobs.crawl_dropout, "company_id"].nunique() / max((jobs.company_id.value_counts() >= BATCH_MIN).sum(), 1)),
}
del codes, first_day, last_day

# Memory: every cleaned listing (reposts flagged) goes to disk for section 8; jobs keeps only the columns the
# repost and file-order checks below need, and jo (originals) is read back from disk.
CLEANED_PATH = CLEAN_DIR / "cleaned_rows_full.parquet"
jobs.to_parquet(CLEANED_PATH)
JOBS_COLS = ["company_id", "first_seen", "occupation_group", "onet_code", "is_repost", "language", "country", "has_desc",
             "mentions_remote", "contract_bits"]
jobs = jobs[JOBS_COLS]
gc.collect()
jo = pd.read_parquet(CLEANED_PATH, filters=[("is_repost", "==", False)])
log("6.7", "postings in the analysis table (originals only)", len(jo))

In [ ]:
# 6.7b Cleaning report and analysis table at a glance
show("Cleaning report (every action, in order)", pd.DataFrame(report).assign(
    **{"share of start": lambda d: (d.rows / report[0]["rows"]).round(4)}))
start_rows = report[0]["rows"]
show("From raw to analysis table", pd.DataFrame({
    "rows": {"cleaned rows (6.1)": start_rows, "after date checks (6.3) and duplicates (6.4)": len(jobs),
             "originals, used for the EDA (6.7)": len(jo)},
}).assign(**{"share of start": lambda d: (d.rows / start_rows).round(3)}))
show("Analysis table at a glance", pd.DataFrame({"value": {
    "postings": compact(len(jo)),
    "companies": compact(jo.company_id.nunique()),
    "occupation groups": jo.occupation_group.nunique(),
    "still open on September 30": f"{jo.live.mean():.1%}",
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
}}))
section_done("6. Cleaning (every listing)")

**What the data shows**
- **Cleaning removed no listings.** It flagged 1.4M reposts, blanked 349K posting dates and 984 pay values.
- **11.7M original listings from 324K companies** feed the every-field comparison (section 7); 22.3% were still open on September 30.

### 6.8 Crawl artifacts (flagged, kept)

Two ways the crawler, not the employer, shapes the dates (measured on the hackathon app's corpus too):
- **Batch first sightings:** when PredictLeads adds a company to its crawl, every listing already on its site is "first seen" the same day. That's not a hiring surge, but it inflates company volume, weekly arrivals and freshness.
- **Crawl dropouts:** when the crawler loses a site, its listings "disappear" the same day. That's not jobs closing, and it cuts days open short.

Both are flags; nothing is removed. Section 9 shows the timing results with and without them.

In [ ]:
# 6.8 Crawl artifacts: how common they are (every cleaned listing)
show("6.8 Crawl artifacts (flagged, kept)", pd.Series(artifact_stats).to_frame("share"))

**What the data shows**
- **Crawl artifacts are small but concentrated:** 2.4% of listings were first seen in a company's one-day batch (17% of companies with 5+ listings have one), and 1.2% disappeared in a one-day dropout (9.9% of those companies).
- **They distort single companies more than the totals,** so they're flagged, never dropped; 9.1 shows timing with and without them.

## 7. Q3: How do computer and mathematical roles compare with every other field?

**Every field, before the computer and mathematical filter (8c).** Before narrowing to the scope, one look at the whole market: every occupation group on the measures a career changer cares about, with computer and mathematical highlighted. Original listings, English only (the language rule of section 8; the stream already kept only US or no-country listings). Two things from every field are saved for later notebooks (7.2).

**Read it as context, not cause:** fields differ in company size, source mix and seniority, so "tech closes faster" can partly reflect who posts tech jobs. Like-for-like comparisons happen inside the scope (EDA_SDA_2 and 4).

### 7.0 Helpers

In [ ]:
# 7.0a Shared constants; full weeks for weekly rates
COMPANY = "company_id"
TOP = 15
SCOPE_GROUP = "Computer and Mathematical"
# Weekly rates use full weeks only: the first week holds the crawler's start-up backlog (listings already up when it
# began), and the last is cut off by the data's end date (Wednesday, September 30). The listings stay; only per-week figures
# leave those two weeks out.
week = jo.first_seen.dt.tz_localize(None).dt.to_period("W")
FIRST_WEEK, LAST_WEEK = week.min(), week.max()
jo["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)
FULL_WEEKS = week[jo.full_week].nunique()
del week
in_market = jo.language.eq("en").to_numpy()  # section 8's language rule
print(f"full weeks: {FULL_WEEKS} (first week {FIRST_WEEK}, last week {LAST_WEEK} left out of weekly rates)")

In [ ]:
# 7.0b Small helpers
def as_list(v):
    """Nested list fields come back as lists or numpy arrays; normalize to a list."""
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    return [] if v is None or (isinstance(v, float) and np.isnan(v)) else [v]

def pct(x):
    return f"{100 * x:.1f}%"

def bar_highlight(ax, s, title, xlabel, fmt, highlight):
    """Horizontal bars, largest at the top, one bar highlighted."""
    s = s.dropna().sort_values()
    ax.barh([str(i)[:40] for i in s.index], s.values, height=0.65,
            color=[ORANGE if i == highlight else BLUE for i in s.index])
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=INK_2, fontsize=8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.18)

In [ ]:
# 7.0c Kaplan–Meier survival helpers
def km_curve(days, finished):
    """Kaplan–Meier curve: share of postings still open after t days, indexed by t.
    Live postings count as censored, so the open ones don't bias the window short."""
    d = pd.DataFrame({"t": days, "e": finished}).dropna()
    by_t = d.groupby("t")["e"].agg(["size", "sum"])
    at_risk = len(d) - by_t["size"].cumsum().shift(fill_value=0)
    return (1 - by_t["sum"] / at_risk).cumprod()

def surv_at(curve, x):
    return curve[curve.index <= x].iloc[-1] if (curve.index <= x).any() else 1.0

def km_survival(days, finished, at):
    curve = km_curve(days, finished)
    return [surv_at(curve, x) for x in at]

def km_median(days, finished):
    """Days until half of postings have closed; NaN if more than half are still open."""
    curve = km_curve(days, finished)
    return curve[curve <= 0.5].index.min()

### 7.1 Every field side by side

| Measure | What it tells a career changer |
|:--|:--|
| Share of all listings, new per week (full weeks) | How big the field is |
| Median hourly pay; pay disclosed | Whether a move pays, and how often pay is shown |
| Median days open (Kaplan–Meier); closed within 7 days | How fast to act |
| Entry-level seniority; asks 2 years or fewer; open to career changers (any sign) | How reachable the field is for a newcomer |
| Degree required; mentions remote (listings with a description) | Two common blockers or draws |

In [ ]:
# 7.1a Every field side by side (original listings, English; text measures among listings with a description)
FIELD_COLS = ["occupation_group", COMPANY, "days_open", "live", "salary_hourly_usd", "entry_seniority", "career_open",
              "years_required", "degree_required", "mentions_remote", "has_desc", "full_week"]
market = jo.loc[in_market, FIELD_COLS]

def field_stats(g):
    curve = km_curve(g.days_open, ~g.live)
    d = g[g.has_desc]
    stated = g.years_required.dropna()
    return pd.Series({
        "listings": len(g),
        "share of all listings": len(g) / len(market),
        "new per week": g.full_week.sum() / FULL_WEEKS,
        "companies": g[COMPANY].nunique(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median days open (Kaplan–Meier)": curve[curve <= 0.5].index.min(),
        "closed within 7 days": 1 - surv_at(curve, 7),
        "entry-level seniority": g.entry_seniority.mean(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "open to career changers (any sign)": g.career_open.mean(),
        "degree required": d.degree_required.mean(),
        "mentions remote": d.mentions_remote.mean(),
    })

with warnings.catch_warnings():  # a field with no disclosed pay gives an empty median: NaN, as intended
    warnings.simplefilter("ignore", RuntimeWarning)
    fields = market.groupby("occupation_group", observed=True).apply(field_stats, include_groups=False)
fields = fields.drop(index="(missing)", errors="ignore").sort_values("listings", ascending=False)
fields.insert(0, "scope", np.where(fields.index == SCOPE_GROUP, "◀ scope", ""))
show_shaded(f"Q3a. Every field side by side ({len(fields)} occupation groups; original English listings; each column shaded on its own scale)",
            fields, highlight=SCOPE_GROUP)

measures = fields.drop(columns="scope")
if SCOPE_GROUP in measures.index:
    show(f"Q3a. Where computer and mathematical stands among the {len(measures)} fields", pd.DataFrame({
        "computer and mathematical": measures.loc[SCOPE_GROUP],
        "median field": measures.median(),
        "rank (1 = highest)": measures.rank(ascending=False, method="min").loc[SCOPE_GROUP].astype(int),
    }), by="rows")

In [ ]:
# Q3a heat map: every field on each measure (each column on its own scale; darker = higher; computer and mathematical outlined)
fig, ax = plt.subplots(figsize=(13, 0.36 * len(fields) + 2.2))
column_heatmap(ax, fields.drop(columns=["scope", "listings", "companies"]), "Every field on each measure (darker = higher, per column)",
               highlight=SCOPE_GROUP)
finish()

In [ ]:
# Q3a chart: openness to career changers vs. pay, every field (bubble size = listings; computer and mathematical highlighted)
pts = fields
x, y = 100 * pts["open to career changers (any sign)"], pts["median hourly pay (USD)"]
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.scatter(x, y, s=40 + 900 * pts.listings / pts.listings.max(), alpha=0.8, edgecolor=SURFACE,
           color=[ORANGE if n == SCOPE_GROUP else BLUE for n in pts.index])
for name in pts.index[:12].union([SCOPE_GROUP]).intersection(pts.index):
    ax.annotate(str(name)[:30], (x[name], y[name]), xytext=(6, 4), textcoords="offset points", fontsize=8,
                color=INK if name == SCOPE_GROUP else INK_2, weight="bold" if name == SCOPE_GROUP else "normal")
ax.axvline(100 * market.career_open.mean(), color=MUTED, ls="--", lw=1)
ax.axhline(market.salary_hourly_usd.median(), color=MUTED, ls="--", lw=1)
ax.set(title="Open to career changers vs. pay, every field (dashed: all listings; orange: computer and mathematical)",
       xlabel="% of listings with any sign of being open to career changers", ylabel="median hourly pay ($)")
finish()

In [ ]:
# Q3a charts: how fast listings close, and how many are entry level, by field (computer and mathematical highlighted)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6.5))
bar_highlight(ax1, fields["median days open (Kaplan–Meier)"], "Median days open (Kaplan–Meier)", "days", "{:.0f} d", SCOPE_GROUP)
bar_highlight(ax2, 100 * fields["entry-level seniority"], "Entry-level listings", "% of listings", "{:.0f}%", SCOPE_GROUP)
finish()

**What the data shows**
- **Computer and mathematical is the best-paid and most remote field:** a median **$66.73 an hour** vs. $26.44 for the median field, and 33% of listings mention remote work vs. 9.5%. It's the 4th largest (8.1% of English listings, about 34K new a week).
- **It's also among the hardest to enter:** 9.4% entry level (22nd of 23 fields), **17% of listings that state years ask for 2 or fewer (last of 23; median field 52%)**, and 21% show any sign of openness (22nd). The most open fields (food preparation, building and grounds, transportation) pay $18–22 an hour.
- **Its listings close at an average pace** (median 23.4 days, 14th), though 30% are gone within a week (4th). Worth the move, hard to enter: exactly where hiding what a newcomer can't get, and ranking the rest, helps.

### 7.2 Saved from every field, before the cut

Two summaries need every field, so they're saved here; everything after section 8 is the scope only.
- **Occupation summary:** pay, timing and openness per occupation (original English listings), for the compass's "your pay increase" (pay in the user's current field) and for comparisons. `data/clean/occupation_summary.pkl`.
- **Company activity:** each company's listings in any field and its newest listing date (all cleaned listings), for company volume and the "company still posting" ghost signal (EDA_SDA_2 3.1). `data/clean/company_activity.parquet`. Each listing's count of its company's earlier listings (`company_earlier`) is already a column (6.7a).

In [ ]:
# 7.2a Occupation summary (every field; aggregates only, not shown here) and company activity (every field)
def occ_stats(g):
    return pd.Series({
        "listings": len(g),
        "companies": g[COMPANY].nunique(),
        "pay disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
        "entry-level seniority": g.entry_seniority.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    occupation_summary = (jo.loc[in_market, ["occupation_group", "occupation", COMPANY, "salary_hourly_usd", "days_open",
                                             "live", "entry_seniority", "career_open"]]
                          .groupby(["occupation_group", "occupation"], observed=True).apply(occ_stats, include_groups=False))
OCC_SUMMARY_PATH = CLEAN_DIR / "occupation_summary.pkl"
occupation_summary.to_pickle(OCC_SUMMARY_PATH)

company_activity = jobs.groupby(COMPANY, observed=True).agg(listings_any_field=("first_seen", "size"),
                                                            newest_listing_any_field=("first_seen", "max"))
COMPANY_ACTIVITY_PATH = CLEAN_DIR / "company_activity.parquet"
company_activity.to_parquet(COMPANY_ACTIVITY_PATH)
del market
gc.collect()
print(f"saved {compact(len(occupation_summary))} occupations in {occupation_summary.index.get_level_values(0).nunique()} groups "
      f"({OCC_SUMMARY_PATH.name}) and {compact(len(company_activity))} companies ({COMPANY_ACTIVITY_PATH.name})")
section_done("7. Every field compared, summaries saved")

## 8. The filters: where the data narrows to the roles TripleTen trains for

**8b** applies language and country; **8c is the scope filter:** the step where 13.0M listings in every field become the roles TripleTen trains for, plus adjacent tech roles. Everything before 8c covers every field; everything after it is the scope only.

These decisions were made on the first 200,000 documents (and the scope on 2026-10-09/10), so they're applied before any further exploration: every later table describes the people we're building for. Nothing is deleted: the 13.0M cleaned rows stay on disk, and section 7 has already summarized every field.

| Decision | Choice | Why | Applied |
|:--|:--|:--|:--|
| **Language** | English only | The text features and the app are English | 8b |
| **No language label** | Drop | Can't be confirmed as English; almost all have no description (8a) | 8b |
| **Country** | United States only: **listings that name the US**; every listing with no country is dropped, including those from US-based companies | Focus solely on US openings: a listing counts only if it says it's in the US. This drops some US remote roles, which often have no location (8a). Decided 2026-10-12, replacing option B (2026-10-10: no-country listings kept when the company is US-based) | The stream (US or no country) and 8b |
| **Field: the scope** | **Roles TripleTen trains for, plus adjacent tech roles:** computer and mathematical (by occupation group or a 15- O*NET code), plus sales engineers (technical sales: TripleTen's Forward Deployed Engineering and AI Automation) and management and market research analysts (business and data analysis: TripleTen's Data Analytics) | Decided 2026-10-09 (computer and mathematical); on 2026-10-10 anchored to TripleTen's programs, a neutral outside definition rather than any team member's background (IT managers and financial analysts out: borderline); section 7 shows how the group compares | **8c** |
| **Credentialed math** | Out (actuaries, mathematicians) | Credential-gated, and no career switches into it in the profiles | **8c** |
| **Time horizon** | April–September by month; the data's first two days (late March) kept, flagged `partial_month` | Hide, never remove: they're left out of per-month counts only | 8b |
| **Seniority** | Keep every level, labelled | Manager-level listings are hidden per applicant (EDA_SDA_2), not here | – |
| **Follow-up window** | Keep each listing's full record | A long-open listing could be a ghost or a hard-to-fill role; EDA_SDA_2 separates them | – |
| **Additional cleaning** | In EDA_SDA_2 (2b) and EDA_SDA_3 | Staffing agencies, salary sanity, internships, multi-location listings, description cleanup: fix or flag, never drop | – |

### Prioritization factors (Q1)

| Factor | Choice | Reason | Status |
|:--|:--|:--|:--|
| Still open / apply-by | The model's **target**, not an input | It's what the ranking predicts from the other factors | Decided |
| Company hiring volume | Keep, counted from the company's listings in **any field** posted **before** each listing (`company_earlier`) | The strongest signal on the sample. High volume could mean **more competition or more openings to win**, so its direction is still open | Kept; interpretation open |
| Source / platform | Keep for now: `source_type` and `platform`; keep job-board rows | Whether source matters beyond company volume, whether platform gaps reflect crawler detection, and whether job-board listings duplicate company-site ones are checked in EDA_SDA_2 | Kept for now |
| Freshness (days since first seen) | Keep for now: `first_seen`, with `posted` alongside | How large the late-found tail is, and whether late-found listings close sooner, are checked in EDA_SDA_2 | Kept for now |
| Pay disclosed | Keep for now, alongside pay level | How much pay-transparency laws and hourly vs. salaried jobs drive it is checked in EDA_SDA_2; also lets pay level be used without dropping listings | Kept for now |
| Pay level | Keep | A direct measure of whether a listing is worth an application | Decided |
| Description length | Keep until EDA_SDA_3 | Length may not be intent | Kept for now |
| Urgency wording | **Left to EDA_SDA_3's NLP** (the column is saved, not analyzed here) | The keyword flag misfires ("urgent care") and showed no gap in days open on the sample; reading the wording in context can tell real urgency apart | Moved to EDA_SDA_3 |
| Remote mentions | Keep | Differs widely between fields and matters to applicants | Decided |
| Role bucket | Keep as its own feature (inside the scope the occupation group is one value) | The eight buckets a career changer chooses between; supports showing related roles in the app | Decided |
| Reposts | **Keep as a feature:** `is_repost`, `times_listed`, `days_since_first_listed`, `days_since_repost_last_listed` | A job posted again and again may be hard to fill. Built from earlier copies only, so known when a listing appears | Decided |
| Hiring difficulty | Keep: age so far, `times_listed`, `days_since_repost_last_listed` | A role open for long or relisted may be hard to fill; EDA_SDA_2 first separates real difficulty from ghost listings (done in 3.1: likely ghosts are hidden, other listings open 90+ days are labelled, and "possibly hard to fill" was dropped) | Kept for now |
| Career-change accessibility | Keep: `entry_seniority`, `years_required`, `low_experience_ask`, `entry_wording`, `changer_wording`, `degree_required`, `equivalent_experience`, `career_open` | Core to the project: hard blockers **hide** a listing; softer signals raise it. From seniority and listing text, so known when a listing appears (no leakage). Keyword rules, checked in EDA_SDA_3 | Decided; rules validated in EDA_SDA_3 |
| Recruiters | Keep: `recruiter_key` (anonymous), `recruiter_title` | A recruiter who fills roles fast or posts on a pattern is timing information; names and contacts are never kept | Kept for now (Q1h) |
| Transferable fit | Keep for later | Needs career-changer profiles (`Profiles.ipynb`) and O*NET skills | Kept, not yet applicable |

### 8.0 Why this scope: the roles TripleTen trains for, English, United States

| Choice | Why | Evidence |
|:--|:--|:--|
| **The roles TripleTen trains for, plus adjacent tech roles** (computer and mathematical, plus sales engineers (technical sales: TripleTen's Forward Deployed Engineering and AI Automation) and management and market research analysts (business and data analysis: TripleTen's Data Analytics)) | 0. **The scope's heading is TripleTen's program catalog** (tripleten.com, checked 2026-10-10: AI Software Engineering, AI & Machine Learning, Data Analytics, Quality Assurance, Cybersecurity, UX/UI Design, AI Automation, AI Systems Engineering; Forward Deployed Engineering and AI Cybersecurity coming soon). It's a neutral, outside definition: no role is in because of a team member's background. Each listing carries its program (`tripleten_program`); IT support, networks and systems, database and IT project management have no program but stay as **adjacent tech roles**, since IT support is the most open entry point and the profiles show real paths through them. Reasons 1–4 are for the core computer and mathematical group. 1. **It's the field we're breaking into ourselves:** the team are TripleTen students retraining into tech, so this was the field that interested us most and that we understand from the inside, and it fits the project brief (career changers moving into computer and mathematical roles). **The data then had to support it as a viable scope, and it does:** 2. **A switch pays off there:** higher pay, more remote work, and a large, steady flow of openings (section 7 ranks the field on each). 3. **Real paths in exist:** 2,813 career changers in the resume data moved into these roles (`Profiles.ipynb`), so the compass can pair the people who switched with the 2026 openings. 4. **It's hard to enter** (few entry-level listings), which is exactly where hiding what a newcomer can't get and ranking the rest helps | 8.0 table below; section 7 |
| **English only** | The text features (EDA_SDA_3), the matching models (ML_Fit) and the app read English. Listings with no language label almost never have a description, so there's nothing to read | 8.0 table below; 8a |
| **United States** | The project focuses solely on US jobs: pay is in dollars, job-growth projections (BLS) are for the US, and mixing in foreign jobs would confound pay and timing. The data is effectively US-only already (on the 200,000-row sample, every listing with a country was in the US). Listings with **no country** are dropped, even when their company is US-based (decided 2026-10-12), though they're more often remote (8a) | 8.0 table below; 8a |

**One limit:** the stream (6.1) removed other languages and countries in one step, so at full scale the share it removed can't be split between "not English" and "not US"; on the sample, language accounted for nearly all of it.

In [ ]:
# 8.0 Evidence for the scope (every field; shares and ranks only)
from src.career_switch import EXTRA_ORDER
US_COMPANIES = set(companies.loc[companies.country.eq("United States"), "id"])  # companies based in the US (8b)
ranks = measures.rank(ascending=False, method="min")
n_fields = len(measures)
def tech_rank(col):
    return ranks.at[SCOPE_GROUP, col] if SCOPE_GROUP in ranks.index else np.nan
no_label = jobs.language.isna()
evidence = pd.Series({
    f"computer and mathematical: rank on median hourly pay (of {n_fields} fields; 1 = highest)": tech_rank("median hourly pay (USD)"),
    f"computer and mathematical: rank on mentions remote (of {n_fields})": tech_rank("mentions remote"),
    f"computer and mathematical: rank on listings (of {n_fields})": tech_rank("listings"),
    f"computer and mathematical: rank on entry-level seniority (of {n_fields}; a high number = few entry-level roles)": tech_rank("entry-level seniority"),
    "computer and mathematical: share of all English listings": fields.at[SCOPE_GROUP, "share of all listings"] if SCOPE_GROUP in fields.index else np.nan,
    "English: share of cleaned listings": jobs.language.eq("en").mean(),
    "no language label: share of cleaned listings": no_label.mean(),
    "no language label: share with no description": (no_label & ~jobs.has_desc.fillna(False).astype(bool)).sum() / max(no_label.sum(), 1),
    "removed by the stream (another language or country): share of the file": 1 - stream["rows"] / stream["records"],
    "country outside the US after the stream: share of cleaned listings": (jobs.country.notna() & jobs.country.ne("United States")).mean(),
    "no country: share of cleaned listings": jobs.country.isna().mean(),
    "no country, company based in the US (dropped): share of no-country listings": (
        jobs.country.isna() & jobs[COMPANY].isin(US_COMPANIES)).sum() / max(jobs.country.isna().sum(), 1),
})
show("8.0 Evidence for the scope", evidence.to_frame("value"), by="rows")

**What the data shows**
- **The evidence supports the scope:** computer and mathematical ranks 1st of 23 fields on pay and on remote work, 4th on volume and 22nd on entry-level roles.
- **Language and country remove little:** 95% of cleaned listings are English; the 4.8% with no language label have essentially no description (623K of 624K); no listing outside the US remains after the stream.
- **8.0% have no country, and 93% of those come from a US-based company** (all dropped, decided 2026-10-12).

### 8a Checks before the filters

Two quick checks that can change which rows the scope keeps (every cleaned listing).
- **Missing country vs. remote:** remote listings often have no location. If the listings with no country are mostly remote, dropping them removes many US remote jobs; 8b drops them anyway (decided 2026-10-12), since only a listing that names the US is known to be a US opening.
- **Where the no-country listings' companies are based:** the share with a US-based company, a company elsewhere, or a company with no location (all dropped).
- **Missing language vs. missing description:** PredictLeads probably detects language from the description. If the unlabeled listings are mostly the ones without a description, dropping them is really a "has description" filter.

In [ ]:
# 8a Check: do listings with no country mention remote work more often? And where are their companies based?
REMOTE_BITS = sum(1 << i for i, label in enumerate(labels["contract_types"][:63]) if "remote" in label)  # "fully remote", "100% remote"...
remote_contract = pd.Series((jobs.contract_bits.to_numpy() & REMOTE_BITS) != 0, index=jobs.index)
show("Remote mentions, by whether the country is known", pd.DataFrame({
    "country_known": jobs.country.notna(), "mentions_remote": jobs.mentions_remote, "remote_contract_type": remote_contract,
}).groupby("country_known").agg(
    listings=("mentions_remote", "size"),
    mentions_remote=("mentions_remote", "mean"),
    remote_contract_type=("remote_contract_type", "mean"),
))
co_country = cat_map(jobs[COMPANY], companies.set_index("id").country.eq("United States").astype(float))
no_country = jobs.country.isna()
show("8a. Listings with no country: where their companies are based", pd.DataFrame({"share of no-country listings": {
    "company based in the US": (no_country & co_country.eq(1)).sum() / max(no_country.sum(), 1),
    "company based elsewhere (dropped)": (no_country & co_country.eq(0) & jobs[COMPANY].map(
        companies.set_index("id").country).notna().to_numpy()).sum() / max(no_country.sum(), 1),
    "company location unknown or not in the companies table (dropped)": (no_country & ~co_country.eq(1) & jobs[COMPANY].map(
        companies.set_index("id").country).isna().to_numpy()).sum() / max(no_country.sum(), 1),
}}))

In [ ]:
# 10a Check: are listings with no language label mostly the ones without a description?
show("Language label vs. description (listings)",
     pd.crosstab(jobs.language.notna().map({True: "language known", False: "no language label"}),
                 jobs.has_desc.map({True: "has description", False: "no description"}), margins=True))

**What the checks show**
- **Listings with no country are largely US remote jobs:** 30% mention remote work vs. 13% with a country, and they use a remote contract label more often (0.83% vs. 0.51%). 93% come from a US-based company, 2.6% from a company elsewhere and 4.3% from an unknown one; all are dropped (decided 2026-10-12). The counts above were measured when US-based companies' no-country listings were still kept (2026-10-10); rerun to refresh them.
- **No language label means no description:** 623K of the 624K unlabeled listings have none, so dropping them also removes about 28% of all listings without a description (623K of 2.2M).

### 8b Language and country

English only (listings with no language label dropped); United States only: listings that name the US; every listing with no country is dropped (decided 2026-10-12). Counted on every cleaned listing, in every field.

In [ ]:
# 8b Language and country: counted on every cleaned listing (every field)
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
from src.career_switch import EXTRA_ORDER, PROGRAM_ORDER, ROLE_ORDER, role_bucket, tripleten_program

FILTERS = {"language": "en", "country": "United States", "first_month": "2026-04"}
english = jobs.language.eq(FILTERS["language"])
names_us = jobs.country.eq(FILTERS["country"])
us_company = jobs[COMPANY].isin(US_COMPANIES)  # the company's first location is in the US (companies table); shown, not used
steps = [("listings in the full file", stream["records"]),
         ("English or unlabeled, US or no country (the stream, 6.1)", stream["rows"]),
         ("cleaned: impossible dates and duplicate ids removed (6.3, 6.4)", len(jobs)),
         ("English only", english.sum()),
         ("United States only: the listing names the US", (english & names_us).sum())]
no_country = english & jobs.country.isna()
show("8b. How English listings place in the US", pd.DataFrame({"share of English listings": {
    "the listing names the US (kept)": (english & names_us).sum() / english.sum(),
    "no country, company based in the US (dropped)": (no_country & us_company).sum() / english.sum(),
    "no country, company elsewhere or unknown (dropped)": (no_country & ~us_company).sum() / english.sum(),
}}))
show("8b. Language and country (every field)", pd.DataFrame(steps, columns=["step", "listings"]).set_index("step")
     .assign(**{"share of the file": lambda d: d.listings / d.listings.iloc[0]}))

### 8c The scope filter: the roles TripleTen trains for, plus adjacent tech roles

**This is the cut.** A listing is in the scope when:
1. its **occupation group** is *Computer and Mathematical*, **or** its **O*NET code starts with 15-** (the same family; this catches listings whose group is missing), **or** it's one of the **two TripleTen roles outside that group:** sales engineers (technical sales: TripleTen's Forward Deployed Engineering and AI Automation) and management and market research analysts (business and data analysis: TripleTen's Data Analytics); and
2. it isn't **credentialed math**: actuaries (15-2011) and mathematicians (15-2021), which need credentials a career changer doesn't have, and which no one in the career-switch profiles moved into.

Every other role is set aside here (it stays on disk, and section 7 already compared every field). The scope is split into **role buckets** by O*NET code (`src/career_switch.py`): the eight computer and mathematical buckets plus the two added ones. **Each listing also gets its TripleTen program** (`tripleten_program`): AI Software Engineering, UX/UI Design, AI & Machine Learning, Data Analytics, Quality Assurance, Cybersecurity, or Forward Deployed Engineering / AI Automation; listings no program trains for (IT support, networks and systems, database, IT project management, other computer roles) are **adjacent tech roles**. IT managers and financial analysts are out (decided 2026-10-10: borderline, a move-up role and a finance-credentialed one). The career-switch profiles (frozen) have paths into the eight only, so for the added buckets the compass shows 2026 openings, pay and timing without past switches. From here on, every table is the scope only.

In [ ]:
# 8c The scope filter (the cut): roles TripleTen trains for, plus adjacent tech roles
english_us = english & names_us
by_group = jobs.occupation_group.eq(SCOPE_GROUP)
by_code = cat_contains(jobs.onet_code, r"^15-")
bucket = relabel(jobs.onet_code, role_bucket)  # role bucket from the O*NET code (None outside the mapped codes)
added = bucket.isin(EXTRA_ORDER)  # the two TripleTen roles outside the group (EXTRA_ONET_BUCKETS)
credentialed = bucket.eq("credentialed math")
in_scope = english_us & (by_group | by_code | added) & ~credentialed
show("8c. How English US listings enter the scope", pd.DataFrame({"share of English US listings": {
    "occupation group is computer and mathematical": (english_us & by_group).mean() / english_us.mean(),
    "  of which the O*NET code also starts with 15-": (english_us & by_group & by_code).mean() / english_us.mean(),
    "only the O*NET code starts with 15- (group missing or different)": (english_us & by_code & ~by_group).mean() / english_us.mean(),
    "credentialed math (actuaries, mathematicians): set aside": (english_us & (by_group | by_code) & credentialed).mean() / english_us.mean(),
    "a TripleTen role outside the group (sales engineers, business and market research analysts)": (english_us & added & ~(by_group | by_code)).mean() / english_us.mean(),
    "in the scope": in_scope.mean() / english_us.mean(),
    "every other role: set aside (stays on disk; compared in section 7)": (english_us & ~(by_group | by_code | added)).mean() / english_us.mean(),
}}))
steps += [("TripleTen roles plus adjacent tech roles (computer and mathematical, or an added TripleTen role)", (english_us & (by_group | by_code | added)).sum()),
          ("credentialed math out (actuaries, mathematicians)", in_scope.sum())]
scope_rows = jobs.index[in_scope.to_numpy()].to_numpy()

# Only the scope's rows are read back from disk (every column); every-field tables are no longer needed
del jo, jobs, english, names_us, us_company, english_us, by_group, by_code, bucket, added, credentialed, in_scope, in_market
gc.collect()
final = ds.dataset(CLEANED_PATH).to_table(filter=pc.field("__index_level_0__").isin(pa.array(scope_rows))).to_pandas()
final["partial_month"] = final.month < pd.Period(FILTERS["first_month"], "M")  # hide, never remove
week = final.first_seen.dt.tz_localize(None).dt.to_period("W")
final["full_week"] = (week > FIRST_WEEK) & (week < LAST_WEEK)
final["role_bucket"] = with_missing(relabel(final.onet_code, role_bucket), "(unmapped 15- code)")
final["tripleten_program"] = with_missing(relabel(final.onet_code, tripleten_program), "adjacent tech role")
steps.append((f"first seen before {FILTERS['first_month']}: kept, flagged partial_month ({final.partial_month.mean():.1%})", len(final)))

funnel = pd.DataFrame(steps, columns=["step", "listings"]).set_index("step")
funnel["removed"] = (-funnel.listings.diff()).fillna(0).astype(int)
funnel["share of the file"] = funnel.listings / funnel.listings.iloc[0]
funnel["share removed at this step"] = funnel.removed / funnel.listings.shift().fillna(funnel.listings.iloc[0])
show("8c. From the full file to the scope (every step)", funnel)
buckets = final.role_bucket.value_counts(normalize=True)
show("8c. The scope by role bucket", pd.DataFrame({"share of the scope": buckets[buckets > 0]}))
programs = final.tripleten_program.value_counts(normalize=True).reindex(PROGRAM_ORDER).dropna()
show("8c. The scope by TripleTen program (adjacent tech role = no program trains for it)",
     pd.DataFrame({"share of the scope": programs[programs > 0]}))
unmapped = final.role_bucket.eq("(unmapped 15- code)")
codes = final.onet_code[unmapped].value_counts(normalize=True)
print(f"scope listings whose O*NET code maps to no role bucket: {unmapped.mean():.2%}")
if unmapped.any():
    show("8c. Unmapped O*NET codes in the scope (add them to src/career_switch.py if they're large)",
         codes[codes > 0].head(10).to_frame("share of unmapped listings"))

**What the data shows**
- **The scope is 1.0M listings, 7.5% of the file:** computer and mathematical is 7.5% of English US listings, the two added TripleTen roles add 0.71%, and credentialed math is 0.05%. The O*NET-code fallback adds nothing (the group is always present), and every scope code maps to a role bucket.
- **Software development is the largest bucket (30%),** then networks and systems (18%), IT project management (12%) and data science (11%); technical sales is the smallest (1.0%).
- **By TripleTen program:** AI Software Engineering 29%, Data Analytics 14%, Cybersecurity 4.9%, AI & Machine Learning 4.8%, Quality Assurance 3.6%, UX/UI Design 1.3%, Forward Deployed Engineering / AI Automation 1.0%; **41% are adjacent tech roles.**

### 8d Fields only the scope carries

The stream kept **every original field for the scope listings** (`data/flat/job_openings_full/`), so the scope gets fields the 13.0M cleaned rows don't carry:
- **The full description** of every scope listing, saved next to the scope table in `data/clean/scope_descriptions.parquet` (id and description; EDA_SDA_3 reads it). Listings outside the scope keep only description counts and flags.
- **Recruiters:** an anonymous key per recruiter (`recruiter_key`: a fingerprint of name and company with a random salt made fresh each run and never saved, so names can't be recovered) and the recruiter's job title. Names and contact details are personal data and aren't kept.
- **Pay as written** (`salary_text`), parsed into hourly pay where `salary_data` has none (`pay_from_text`), with the same bounds as 6.5.
- **Location as written** (`location_text`), the **url** (the app links to the listing) and the two record types (kept for completeness).

In [ ]:
# 8d Scope-only fields from the stream's scope parts (every original field); descriptions written to disk
from src.full_rows import ids_at, scope_fields
from src.listing_rows import pay_from_text
start = time.time()
DESC_PATH = CLEAN_DIR / "scope_descriptions.parquet"
ids = ids_at(final.index)
extras = scope_fields(ids, DESC_PATH)
final = final.join(extras)
del extras
gc.collect()

# Pay from the text where salary_data had none (same conversion and bounds as 6.5)
needs = final.salary_hourly_usd.isna() & final.salary_text.notna()
parsed = final.loc[needs, "salary_text"].map(pay_from_text).dropna()
low, high, unit = (parsed.str[k] for k in range(3))
hourly = (low + high) / 2 * unit.map(PER_YEAR) / HOURS_PER_YEAR
hourly = hourly[hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR)]
final["pay_from_text"] = False
final.loc[hourly.index, "salary_hourly_usd"] = hourly.astype("float32")
final.loc[hourly.index, "pay_from_text"] = True
missing = final.record_type.isna()
if missing.mean() > 0.01:
    print(f"NOTE: {missing.mean():.1%} of scope listings aren't in the stream's saved full records. If they're the two added TripleTen "
          "roles, run `python -m src.stream_full --extra` (about an hour), then rerun from 8d.")
show("8d. Scope-only fields: share of scope listings with each", pd.DataFrame({"share": {
    "found in the stream's scope parts": final.record_type.notna().mean(),
    "a description (full text saved)": final.has_desc.mean(),
    "a named recruiter (anonymous key)": final.recruiter_key.notna().mean(),
    "a recruiter title": final.recruiter_title.notna().mean(),
    "pay as written": final.salary_text.notna().mean(),
    "location as written": final.location_text.notna().mean(),
    "hourly pay before 8c": final.salary_hourly_usd.notna().mean() - final.pay_from_text.mean(),
    "hourly pay added from the text": final.pay_from_text.mean(),
    "hourly pay after 8c": final.salary_hourly_usd.notna().mean(),
}}).round(4))
print(f"descriptions saved to {DESC_PATH.name} ({DESC_PATH.stat().st_size / 1e9:.2f} GB) in {time.time() - start:.0f}s")
section_done("8. Filters and scope-only fields")

**What the data shows**
- **Every scope listing was found in the stream's full records;** 78% have a description (2.3 GB of text saved for EDA_SDA_3).
- **Pay as written exists for 45%,** and parsing it added hourly pay for 1.4% of listings (43% to 45%).
- **Recruiters are rare:** 1.2% of listings name one (stored as an anonymous key), 1.1% give their title; 90% carry a location as written.

## 9. Prioritization EDA inside the scope, by role bucket

**From here on: scope listings only** (the scope filter in 8c: the roles TripleTen trains for, plus adjacent tech roles).

Q1 measures every candidate factor across the **role buckets** (the eight computer and mathematical buckets, other computer roles, and the two added TripleTen buckets) and, in Q1a, by **TripleTen program**, Q2 checks which time information the data supports, and 9.3 shows the scope's mix. All use the scope's original listings (`so`); repost rates use every scope listing (`final`).

In [ ]:
# 9.0 The scope's original listings, and the role buckets in display order
so = final[~final.is_repost]
BUCKETS = [b for b in ROLE_ORDER + ["other computer roles"] + EXTRA_ORDER + ["(unmapped 15- code)"] if (so.role_bucket == b).any()]
PROGRAMS = [p for p in PROGRAM_ORDER if (so.tripleten_program == p).any()]
print(f"scope: {compact(len(final))} listings ({compact(len(so))} originals) in {len(BUCKETS)} role buckets")

### 9.1 Q1: Which factors should drive prioritization?

A factor only helps ranking if it **varies**, so Q1a shows each factor across the scope, across the role buckets, and its spread (max ÷ min). Q1b–Q1h look closer.

| Measure | Prioritization factor | What it tells an applicant | Caveat |
|---|---|---|---|
| Median days open; closed within 7 days | Still open / apply-by | How long, and how urgent, the window is | Disappearing ≠ filled |
| New listings per week (full weeks) | Freshness, source | How often to check | Depends on crawl frequency |
| Still open on September 30 | Still open | How much is rankable now | One day only |
| Repost rate | Competition | Jobs that didn't fill | Could also mean a bad job |
| Pay disclosed, median pay | Pay | Whether it's worth applying | Under half disclose pay |
| Mentions remote | Employer intent | Flexible or not | Keyword flag (urgency wording is left to EDA_SDA_3's NLP) |
| From a hiring platform | Source | Where listings are hosted | From the url domain |
| Top 10 companies' share | Competition, source | Whether a short watch list covers most listings | |
| Entry-level seniority; asks 2 years or fewer; open to career changers | Career-change accessibility | Whether a newcomer can get in | Keyword rules; validated in EDA_SDA_3 |

In [ ]:
# 9.1 Q1a: the candidate prioritization factors for one group (g = originals, g_all = originals + reposts)
def targets(g, g_all):
    curve = km_curve(g.days_open, ~g.live)
    top10 = g[COMPANY].value_counts().head(10).sum()
    return pd.Series({
        "median days open (Kaplan–Meier)": curve[curve <= 0.5].index.min(),
        "closed within 7 days": 1 - surv_at(curve, 7),
        "new postings per week": g.full_week.sum() / FULL_WEEKS,
        "still open on September 30 (share)": g.live.mean(),
        "repost rate": g_all.is_repost.mean(),
        "salary disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "mentions remote": g.mentions_remote.mean(),
        "from a hiring platform": (g.source_type == "hiring platform").mean(),
        "top 10 companies' share": top10 / len(g),
        "entry-level seniority": g.entry_seniority.mean(),
        "asks 2 years or fewer": g.low_experience_ask.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })

In [ ]:
# Q1a: the factors across the scope and per role bucket
by_bucket = pd.DataFrame({b: targets(so[so.role_bucket == b], final[final.role_bucket == b]) for b in BUCKETS}).T
overview = pd.DataFrame({
    "whole scope": targets(so, final),
    "bucket min": by_bucket.min(),
    "bucket median": by_bucket.median(),
    "bucket max": by_bucket.max(),
})
overview["spread (max / min)"] = overview["bucket max"] / overview["bucket min"].replace(0, np.nan)
show("Q1a. Candidate prioritization factors: the whole scope, and the range across role buckets", overview, by="rows")
show_shaded("Q1a. Candidate prioritization factors by role bucket (each column shaded on its own scale)", by_bucket)
by_program = pd.DataFrame({p: targets(so[so.tripleten_program == p], final[final.tripleten_program == p]) for p in PROGRAMS}).T
show_shaded("Q1a. The same factors by TripleTen program (adjacent tech role = no program trains for it)", by_program,
            highlight="adjacent tech role")

In [ ]:
# Q1a chart: how much each factor differs between role buckets
fig, ax = plt.subplots(figsize=(8, 4))
bar_h(ax, overview["spread (max / min)"], "How much each factor differs between role buckets",
      "spread: highest bucket ÷ lowest bucket", "{:.1f}×")
ax.axvline(1, color=INK_2, lw=1)
finish()

**What the data shows**
- **Accessibility varies most between buckets:** entry-level seniority 10.8× (2.1% in IT project management to 23% in IT support), open to career changers 4.8× (9.2% to 44%), asks 2 years or fewer 4.7×.
- **Pay varies 2.4×** ($31.00 an hour in IT support to $73.23 in software development) **and volume 29×** (364 new a week in technical sales to 11K in software development), while timing varies only 1.8× (medians of 21–38 days).
- **By program:** Data Analytics roles are the most open (30% show any sign, 13% ask for 2 years or fewer) at $47.64 an hour; AI Software Engineering and AI & Machine Learning pay the most (about $73.50) but are among the least open (18–19%).

#### Q1b. Still open / apply-by

The share of listings still open N days after first being seen, using Kaplan–Meier survival curves. These treat still-open listings as unfinished rather than ignoring them, which would make windows look shorter than they are. This is the basis for an **apply-by estimate** on each listing.

In [ ]:
# 9.1 Q1b: application window by role bucket
DAYS = [3, 7, 14, 30, 60]
window = {b: km_survival(so.days_open[so.role_bucket == b], ~so.live[so.role_bucket == b], DAYS) for b in BUCKETS}
window["(whole scope)"] = km_survival(so.days_open, ~so.live, DAYS)
show_shaded("Q1b. Share of listings still open N days after first seen (Kaplan–Meier)",
            pd.DataFrame(window, index=[f"open after {d}d" for d in DAYS]).T, highlight="(whole scope)")

In [ ]:
# Q1b chart: survival curves for the fastest and slowest role buckets
grid = np.arange(0, 121)
def on_grid(curve):
    return curve.reindex(curve.index.union(grid)).ffill().reindex(grid).fillna(1.0)

medians = {b: km_median(so.days_open[so.role_bucket == b], ~so.live[so.role_bucket == b]) for b in BUCKETS}
medians = {b: m for b, m in medians.items() if pd.notna(m)}
fast, slow = min(medians, key=medians.get), max(medians, key=medians.get)
fig, ax = plt.subplots(figsize=(9.5, 4.2))
for b in BUCKETS:
    if b not in (fast, slow):
        g = so[so.role_bucket == b]
        ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=MUTED, lw=1)
ax.plot([], [], color=MUTED, lw=1, label="other role buckets")
for b, color, tag in ((fast, ORANGE, "fastest"), (slow, BLUE, "slowest")):
    g = so[so.role_bucket == b]
    ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=color, label=f"{b} ({tag})")
ax.step(grid, on_grid(km_curve(so.days_open, ~so.live)), where="post", color=INK, lw=1.5, ls="--", label="whole scope")
ax.set(title="Share of listings still open, by days since first seen", xlabel="days since first seen",
       ylabel="share still open", xlim=(0, 120), ylim=(0, 1.02))
ax.legend(fontsize=8.5)
finish()

In [ ]:
# Q1b check: timing with and without crawl artifacts (scope originals; 6.8)
views = {"every listing": pd.Series(True, index=so.index),
         "without batch first sightings": ~so.batch_first_seen,
         "without crawl dropouts": ~so.crawl_dropout,
         "without either": ~so.batch_first_seen & ~so.crawl_dropout}
show("Q1b check: timing with and without crawl artifacts", pd.DataFrame({name: {
    "share of listings": m.mean(),
    "median days open (Kaplan–Meier)": km_median(so.days_open[m], ~so.live[m]),
    "open after 7d": km_survival(so.days_open[m], ~so.live[m], [7])[0],
    "open after 30d": km_survival(so.days_open[m], ~so.live[m], [30])[0],
} for name, m in views.items()}).T)

**What the data shows**
- **The window is short:** 29% of scope listings close within a week, the median is 24.1 days, and 45% are still open after 30 days.
- **Most buckets move alike** (66–74% still open after a week). **Technical sales is the slow outlier** (83% after a week, median 38 days); database is the fastest (median 21 days).
- **Crawl artifacts barely move timing:** without them the median falls from 24.1 to 22.4 days, and the share open after a week stays at 70–71%.

#### Q1c. Source

Which hiring platforms, job boards or company sites host the scope's listings, and whether arrival rate and window differ by source. The crawl covers company sites and platforms, so job boards are under-represented.

In [ ]:
# 9.1 Q1c: sources: share of postings, arrival rate, window
show("Q1c. Source type", pd.DataFrame({"share": so.source_type.value_counts(normalize=True).round(3)}))
source = so.platform.astype(object).fillna(so.source_type.astype(object)).astype("category")  # platform, else source type
by_src = so[["days_open", "live", "full_week"]].groupby(source, observed=True).apply(lambda g: pd.Series({
    "postings": len(g),
    "share": len(g) / len(so),
    "new per week": g.full_week.sum() / FULL_WEEKS,
    "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
    "open after 7d": km_survival(g.days_open, ~g.live, [7])[0],
}))
show("Q1c. Platforms and other sources: volume, arrival rate, window", by_src.sort_values("postings", ascending=False).head(TOP))

In [ ]:
# Q1c: source mix within each role bucket
mix = pd.crosstab(so.role_bucket, source, normalize="index").reindex(BUCKETS)
show_shaded("Q1c. Source mix within each role bucket (share of the bucket's listings)",
            mix[mix.sum().sort_values(ascending=False).index[:8]])

In [ ]:
# Q1c chart: median days open by source
fig, ax = plt.subplots(figsize=(8.5, 4.6))
top_src = by_src.sort_values("postings", ascending=False).head(TOP)
bar_h(ax, top_src["median days open (Kaplan–Meier)"], "Median days a listing stays open, by source (15 largest)", "days (Kaplan–Meier median)", "{:.0f} d")
finish()

**What the data shows**
- **65% of scope listings are on company sites or other domains,** 28% on hiring platforms and 7.1% on job boards.
- **Windows differ widely by source:** LinkedIn, Oracle and Taleo listings close in a median 15 days, company sites 21, Workday 32, Greenhouse 49, Lever 66 and Ashby 71.
- **The mix is similar in every bucket** (company sites 56–70%), so source separates listings, not roles. Platform choice tracks employer size, so source is tested alongside company volume.

#### Q1d. Competition: company hiring volume

Whether a few companies post most listings (a short watch list for an applicant), and whether listings from high-volume companies close faster or slower. **Volume counts a company's listings in any field** (section 7.2), since a bank posting 2,000 jobs, 50 of them in tech, is a high-volume employer. Bands: 1–4, 5–49, 50–499 and 500+ listings (rescaled for the full file: the sample's 20+ band held 89% of listings).

In [ ]:
# 9.1 Q1d: company concentration by role bucket (scope listings)
def concentration(g):
    per_co = g[COMPANY].value_counts()
    per_co = per_co[per_co > 0]  # company_id is a category: drop companies with no listings here
    return pd.Series({
        "companies": per_co.size,
        "postings per company (median)": per_co.median(),
        "top 1% of companies' share": per_co.head(max(1, per_co.size // 100)).sum() / len(g),
        "top 10% of companies' share": per_co.head(max(1, per_co.size // 10)).sum() / len(g),
        "companies with 5+ postings": (per_co >= 5).sum(),
    })
show_shaded("Q1d. Company concentration by role bucket",
            so[[COMPANY, "role_bucket"]].groupby("role_bucket", observed=True).apply(concentration, include_groups=False).reindex(BUCKETS))

In [ ]:
# Q1d: do high-volume companies' listings close faster or slower? (volume: listings in any field)
# Bands rescaled for the full file (2026-10-10): with every field counted, 89% of scope listings came from companies with
# 20+ listings, so the sample's 1 / 2-4 / 5-19 / 20+ bands no longer split anything.
BANDS = ["1-4", "5-49", "50-499", "500+"]
def size_band(n):
    return pd.cut(n.fillna(1), [0, 4, 49, 499, np.inf], labels=BANDS)
any_field = cat_map(so[COMPANY], company_activity.listings_any_field)
by_band = so[["days_open", "live"]].groupby(size_band(any_field), observed=True).apply(lambda g: pd.Series({
    "share of postings": len(g) / len(so),
    "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
}))
by_band["repost rate"] = final.is_repost.groupby(size_band(cat_map(final[COMPANY], company_activity.listings_any_field)), observed=True).mean()
show("Q1d. Do high-volume companies' listings close faster or slower? (company's listings in any field, whole six months)", by_band)

In [ ]:
# Q1d check: volume counted from earlier listings only (any field), the way the model will see it
# The table above counts each company's listings over all six months, including later ones (descriptive only).
# Here each listing's band counts only the company's listings first seen before it (company_earlier, 6.7a).
asof_band = size_band(so.company_earlier + 1)
since_may = so.first_seen >= pd.Timestamp("2026-05-01", tz="UTC")  # a month or more of history behind each

def band_medians(mask):
    return so.loc[mask, ["days_open", "live"]].groupby(asof_band[mask], observed=True).apply(lambda g: km_median(g.days_open, ~g.live))

show("Q1d check: median days open (Kaplan–Meier) by company volume, whole six months vs. earlier listings only", pd.DataFrame({
    "median days, whole six months (descriptive)": by_band["median days open (Kaplan–Meier)"],
    "median days, earlier listings only": band_medians(pd.Series(True, index=so.index)),
    "median days, earlier only, first seen May or later": band_medians(since_may),
    "share of listings (earlier only)": asof_band.value_counts(normalize=True),
}).reindex(BANDS))

In [ ]:
# Q1d chart: survival curves by company volume (earlier listings only), with a Gehan–Wilcoxon test of 500+ vs. 1-4 listings
# Each curve is the Kaplan–Meier share of listings still open after each day; a median is where a curve crosses 50%.
# Days open is censored (still-open listings haven't finished), so the Wilcoxon test used here is the survival version:
# a log-rank test weighted toward early differences (Gehan–Breslow–Wilcoxon). Formal tests are in EDA_SDA_4; this is a preview.
# The test runs on up to 500,000 random listings per group; the curves use every listing.
from lifelines.statistics import logrank_test
TEST_N = 500_000
fig, ax = plt.subplots(figsize=(9, 4.2))
for band, color in zip(BANDS, [MUTED, AQUA, ORANGE, BLUE]):
    g = so.loc[asof_band == band, ["days_open", "live"]]
    if g.empty:
        continue
    curve = km_curve(g.days_open, ~g.live)
    ax.step(curve.index, 100 * curve.values, where="post", color=color, lw=1.8,
            label=f"{band} (median {km_median(g.days_open, ~g.live):.0f} days, {compact(len(g))} listings)")
ax.axhline(50, color=INK_2, ls="--", lw=0.8)
ax.text(1, 51.5, "median: half still open", color=INK_2, fontsize=8)
a, b = (so.loc[asof_band == band, ["days_open", "live"]] for band in ("500+", "1-4"))
a, b = (g.sample(min(len(g), TEST_N), random_state=0) for g in (a, b))
title = "Share still open by company volume (earlier listings in any field)"
if len(a) and len(b):
    test = logrank_test(a.days_open, b.days_open, event_observed_A=~a.live, event_observed_B=~b.live, weightings="wilcoxon")
    title += (f"\nGehan–Wilcoxon test, 500+ vs. 1-4 listings: statistic {test.test_statistic:,.0f}, "
              f"p {'< 0.001' if test.p_value < 0.001 else f'= {test.p_value:.3f}'}")
ax.set(title=title, xlabel="days since first seen", ylabel="% still open", xlim=(0, 120), ylim=(0, 100))
ax.legend(fontsize=8, frameon=False)
finish()

In [ ]:
# Q1d charts: days open and share of listings, by company hiring volume (any field, whole six months)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
by_band = by_band.reindex(BANDS)
ax1.bar(BANDS, by_band["median days open (Kaplan–Meier)"], color=BLUE, width=0.6)
for x, v in enumerate(by_band["median days open (Kaplan–Meier)"]):
    if pd.notna(v):
        ax1.text(x, v, f"{v:.0f} d", ha="center", va="bottom", color=INK_2, fontsize=8.5)
ax1.set(title="Median days open, by company hiring volume", xlabel="the company's listings in any field", ylabel="days")
ax1.grid(axis="x", visible=False)
ax2.bar(BANDS, 100 * by_band["share of postings"], color=BLUE, width=0.6)
ax2.set(title="Share of scope listings", xlabel="the company's listings in any field", ylabel="% of listings")
ax2.grid(axis="x", visible=False)
finish()

**What the data shows**
- **Company volume is the strongest timing signal:** counting only earlier listings in any field, scope listings at companies with 500+ listings close in a median **10 days, vs. 50 at companies with 1–4** (Gehan–Wilcoxon test, p < 0.001); over the whole six months, 14 vs. 56.
- **The gradient is steady across bands** (50, 35, 28 and 10 days) and holds for listings first seen from May (60 vs. 10 days).
- **High-volume employers dominate:** half of scope listings come from companies with 500+ listings in any field, and the top 10% of companies post 47–73% of each bucket's listings. Reposting rises with volume (1.2% to 7.6%).

#### Q1e. Competition: reposts

A repost is the same job (company, title and description) listed again under a new id. It suggests the first listing didn't fill, which could mean less competition, or a hard-to-fill or unattractive role. Here: how common reposts are by role bucket, and the gap between original and repost.

In [ ]:
# 9.1 Q1e: repost rates by role bucket, and days from the original to each repost
show("Q1e. Repost rate by role bucket", final.groupby("role_bucket", observed=True).is_repost.agg(
    reposts="sum", repost_rate="mean").reindex(BUCKETS))
first = final[~final.is_repost & final.repost_fp.notna()].groupby("repost_fp").first_seen.min()
reposts = final.loc[final.is_repost, ["repost_fp", "first_seen"]].join(first.rename("original_seen"), on="repost_fp")
gap = (reposts.first_seen - reposts.original_seen).dt.total_seconds() / 86400
show("Q1e. Days from the original to each repost", gap.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("days"))

In [ ]:
# Q1e charts: repost rate and repost gap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
bar_h(ax1, 100 * final.groupby("role_bucket", observed=True).is_repost.mean().reindex(BUCKETS), "Repost rate by role bucket",
      "% of listings that are reposts", "{:.1f}%")
ax2.hist(gap.clip(upper=120), bins=40, color=BLUE)
ax2.set(title="Days from original to repost", xlabel="days (capped at 120)", ylabel="reposts")
finish()

**What the data shows**
- **6.1% of scope listings are reposts:** most in technical sales (8.9%) and business and data analysis (8.1%), fewest in quality assurance (4.3%) and IT support (4.6%).
- **Reposts come back fast:** a median 7.6 days after the original, and a quarter within a day, so many are re-listings or crawler refreshes rather than a failed hire.

#### Q1f. Employer intent: text signals (the NLP link)

A first look at whether simple text features go with listings staying open longer or shorter. Each row compares listings with and without the feature. These are keyword flags, not models, and they misfire ("remote patient monitoring" counts as remote), so EDA_SDA_3 checks a sample of matches. Urgency wording is left out here: its keyword flag misfired ("urgent care") and showed no gap, so EDA_SDA_3 reads it with NLP instead. If any of them show a gap, richer text features (TF-IDF, topics, embeddings) are worth modeling as prioritization inputs.

In [ ]:
# 9.1 Q1f: do text features relate to how long postings stay open?
flags = {
    "salary disclosed": so.salary_hourly_usd.notna(),
    "mentions remote": so.mentions_remote,
    "long description (top 25%)": so.desc_words > so.desc_words.quantile(0.75),
    "on a hiring platform": so.source_type == "hiring platform",
}
rows = {}
for name, mask in flags.items():
    known = mask.notna().to_numpy()                  # text flags are unknown without a description
    has = mask.fillna(False).astype(bool).to_numpy()
    with_f = km_survival(so.days_open[has], ~so.live[has], [7, 30])
    without = km_survival(so.days_open[known & ~has], ~so.live[known & ~has], [7, 30])
    rows[name] = {"share of postings": has[known].mean(), "open 7d (with)": with_f[0], "open 7d (without)": without[0],
                  "open 30d (with)": with_f[1], "open 30d (without)": without[1]}
show("Q1f. Text features vs. application window (scope)", pd.DataFrame(rows).T)

In [ ]:
# Q1f chart: share still open after 7 days, with vs. without each feature
feat = pd.DataFrame(rows).T.sort_values("open 7d (with)")
fig, ax = plt.subplots(figsize=(8.5, 3.4))
y = np.arange(len(feat))
ax.hlines(y, feat["open 7d (without)"], feat["open 7d (with)"], color=MUTED, lw=2)
ax.scatter(feat["open 7d (without)"], y, color=INK_2, s=60, zorder=3, label="without the feature")
ax.scatter(feat["open 7d (with)"], y, color=BLUE, s=60, zorder=3, label="with the feature")
ax.set_yticks(y, feat.index)
ax.set(title="Share of listings still open after 7 days", xlabel="share still open")
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8.5, loc="lower right")
finish()

**What the data shows**
- **Disclosed pay, long descriptions and hiring platforms go with listings staying open longer:** 79–82% still open after a week, vs. 64–67% without.
- **Remote wording shows no gap** (77% vs. 79%).
- **One feature at a time:** these overlap with company volume and source, so EDA_SDA_4's Cox model separates them.

#### Q1g. Career-change accessibility

How open each role bucket is to someone new to the field, and whether open listings behave differently. Text measures use listings with a description.

| Measure | What it tells a career changer | Caveat |
|---|---|---|
| Entry-level seniority | Whether there are roles a newcomer can start in | PredictLeads' label; its rules aren't documented |
| States years required; median years asked; asks 2 years or fewer | How much field experience listings expect | Lower end of each range |
| Entry-level or training wording; career-changer wording | Whether the employer trains or invites newcomers | Keyword rules |
| Degree required; "or equivalent experience" | Whether a missing degree rules the applicant out | One listing can say both |
| Open to career changers (any sign) | The first-pass summary flag | Broad on purpose |
| Median pay | Whether the role is worth the move | Under half disclose pay |

In [ ]:
# 9.1 Q1g: career-change accessibility, by role bucket
def access(g):
    """Career-change accessibility for one group; text measures use listings with a description."""
    d = g[g.has_desc]
    stated = d.years_required.dropna()
    return pd.Series({
        "listings": len(g),
        "entry-level seniority": g.entry_seniority.mean(),
        "states years required": d.years_required.notna().mean(),
        "median years required (where stated)": stated.median(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "entry-level or training wording": d.entry_wording.mean(),
        "career-changer wording": d.changer_wording.mean(),
        "degree required": d.degree_required.mean(),
        "or equivalent experience": d.equivalent_experience.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "median hourly pay (USD), open listings only": g.loc[g.career_open.fillna(False), "salary_hourly_usd"].median(),
    })

ACCESS_COLS = ["has_desc", "entry_seniority", "years_required", "entry_wording", "changer_wording", "degree_required",
               "equivalent_experience", "career_open", "salary_hourly_usd"]
acc = pd.DataFrame({b: access(so.loc[so.role_bucket == b, ACCESS_COLS]) for b in BUCKETS}).T
acc.loc["whole scope"] = access(so[ACCESS_COLS])
show_shaded("Q1g. Career-change accessibility by role bucket (each column shaded on its own scale)", acc, highlight="whole scope")

In [ ]:
# Q1g: do listings open to career changers stay open longer or shorter?
cc_flags = {
    "entry-level seniority": so.entry_seniority,
    "asks 2 years or fewer": (so.years_required <= 2).where(so.years_required.notna()).astype("boolean"),  # only listings that state years
    "asks 5 years or more": (so.years_required >= 5).where(so.years_required.notna()).astype("boolean"),
    "entry-level or training wording": so.entry_wording,
    "degree required": so.degree_required,
    "open to career changers (any sign)": so.career_open,
}
cc_rows = {}
for name, mask in cc_flags.items():
    known = mask.notna().to_numpy()                  # compare with vs. known-without, never vs. unknown
    has = mask.fillna(False).astype(bool).to_numpy()
    with_f = km_survival(so.days_open[has], ~so.live[has], [7, 30])
    without = km_survival(so.days_open[known & ~has], ~so.live[known & ~has], [7, 30])
    cc_rows[name] = {"share of listings": has[known].mean(), "open 7d (with)": with_f[0], "open 7d (without)": without[0],
                     "open 30d (with)": with_f[1], "open 30d (without)": without[1]}
show("Q1g. Career-change flags vs. application window (scope)", pd.DataFrame(cc_rows).T)

In [ ]:
# Q1g chart: how open each role bucket is to career changers, against pay
pts = acc.drop(index="whole scope")
x, y = 100 * pts["open to career changers (any sign)"], pts["median hourly pay (USD)"]
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.scatter(x, y, s=400 * pts["listings"] / pts["listings"].max(), color=BLUE, alpha=0.85, edgecolor=SURFACE)
for name in pts.index:
    ax.annotate(str(name)[:32], (x[name], y[name]), xytext=(6, 4), textcoords="offset points", fontsize=8, color=INK_2)
ax.axvline(100 * acc.loc["whole scope", "open to career changers (any sign)"], color=MUTED, ls="--", lw=1)
ax.axhline(acc.loc["whole scope", "median hourly pay (USD)"], color=MUTED, ls="--", lw=1)
ax.set(title="Open to career changers vs. pay, by role bucket (dashed: whole scope)",
       xlabel="% of listings with any sign of being open to career changers", ylabel="median hourly pay ($)")
finish()

**What the data shows**
- **IT support is the entry door:** 23% entry level, 48% of listings that state years ask for 2 or fewer, and 44% show any sign of openness. Then other computer roles (37%) and business and data analysis (34%). IT project management is the hardest (2.1% entry level, 9.2% open).
- **Open listings pay less, but still well:** a median $37.98 an hour, vs. $64.14 across the scope ($26.00 for open IT support listings).
- **Listings open to career changers close a bit faster** (75% still open after a week vs. 80%). Degree requirements are rare (3.7%), but 60% of listings that state years ask for 5 or more.

#### Q1h. Recruiters

Whether a named recruiter changes how a listing behaves, and whether some recruiters post a lot. Recruiters are an anonymous key only (8d): no names or contacts are kept or shown, and only counts and shares appear here.

In [ ]:
# 9.1 Q1h: recruiters (anonymous keys; aggregates only)
has_rec = so.recruiter_key.notna()
per_rec = so.recruiter_key.dropna().value_counts()
rec_band = pd.cut(cat_map(so.recruiter_key.astype("category"), per_rec) if has_rec.any() else pd.Series(np.nan, index=so.index),
                  [0, 1, 4, np.inf], labels=["1 listing", "2-4", "5+"])
show("Q1h. Recruiters in the scope", pd.DataFrame({"value": {
    "share of listings naming a recruiter": pct(has_rec.mean()),
    "recruiters (anonymous keys)": compact(per_rec.size),
    "recruiters with 5+ listings": compact((per_rec >= 5).sum()),
    "share of recruiter listings from recruiters with 5+": pct(per_rec[per_rec >= 5].sum() / max(per_rec.sum(), 1)),
    "share with a recruiter title": pct(so.recruiter_title.notna().mean()),
}}))
rows = {"named recruiter": has_rec, "no recruiter named": ~has_rec}
rows.update({f"recruiter with {b}": rec_band.eq(b) for b in ["1 listing", "2-4", "5+"]})
show("Q1h. Window by recruiter (Kaplan–Meier)", pd.DataFrame({name: {
    "listings": int(m.sum()),
    "median days open": km_median(so.days_open[m], ~so.live[m]) if m.any() else np.nan,
    "open after 7d": km_survival(so.days_open[m], ~so.live[m], [7])[0] if m.any() else np.nan,
} for name, m in rows.items()}).T)
# the role only: "... at <employer>" and "... @ <employer>" are cut, so no employer names are shown
titles = (so.recruiter_title.str.lower().str.replace(r"\s*(?:\bat\b|@).*$", "", regex=True).str.strip(" -/|,")
          .replace("", pd.NA).dropna().value_counts())
show("Q1h. Most common recruiter titles (the role only; no names or employers)",
     pd.DataFrame({"share of listings with a title": (titles / titles.sum()).head(10)}))

**What the data shows**
- **Only 1.2% of scope listings name a recruiter:** 5,527 anonymous recruiters; the 437 with 5+ listings post 37% of recruiter listings.
- **Recruiter listings close much faster:** a median 11.3 days vs. 24.3 without one; for recruiters with 5+ listings, **6.5 days** (48% still open after a week vs. 71%).
- **The titles point to staffing and talent-acquisition roles** (talent acquisition specialist, technical recruiter, account and delivery managers), so EDA_SDA_2 tests this against its staffing-agency flag before it's used.

### 9.2 Q2: What time information can prioritization rely on?

| Time scale | Use in prioritization | Needs |
|---|---|---|
| Days since a listing appeared | Apply-by estimates and freshness | Closed listings with both dates |
| Week / day of week | How often to check for new listings | Dates that reflect employers, not the crawler |
| Season / month | When to search | About two years of history |

Checks: how far back `first_seen` goes; whether `posted` and `first_seen` weekdays differ (crawler vs. employer); weekly arrivals over **full weeks only** (the first and last weeks are shown greyed and labelled partial). The old check that the file isn't sorted (which would have biased a first-N sample) isn't needed: every listing is used now.

In [ ]:
# 9.2 Q2a: how much history does the data hold?
show("Q2a. How much history does the data hold?", pd.DataFrame({"value": {
    "first_seen from": so.first_seen.min().date(),
    "first_seen to": so.first_seen.max().date(),
    "distinct months": so.month.nunique(),
    "complete years": so.month.nunique() // 12,
    "share already closed by September 30 (high = keeps history)": pct((~so.live).mean()),
}}))
by_month = so.month.value_counts().sort_index()
show("Q2a. Listings by first_seen month", pd.DataFrame({"share": by_month / by_month.sum()}))

In [ ]:
# Q2c: employer or crawler? posted vs. first_seen
both = so.posted.notna()
lag = (so.first_seen - so.posted)[both].dt.total_seconds() / 86400
show(f"Q2c. Days from posted to first_seen (listings with posted: {pct(both.mean())})",
     lag.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = pd.DataFrame({
    "posted (UTC)": so.posted[both].dt.day_name().value_counts(normalize=True),
    # posted timestamps sit at 22:00/23:00 UTC = midnight Central European time, so read them as CET dates
    "posted (CET date)": so.posted[both].dt.tz_convert("Europe/Berlin").dt.day_name().value_counts(normalize=True),
    "first_seen, same listings": so.first_seen[both].dt.day_name().value_counts(normalize=True),
    "first_seen, all listings": so.first_seen.dt.day_name().value_counts(normalize=True),
}).reindex(days)
show("Q2c. % of listings by weekday (if the columns disagree, first_seen reflects the crawler)",
     (100 * weekday).round(1))

In [ ]:
# Q2d: weekly arrivals, full weeks only (the first and last weeks are partial: start-up backlog, and cut off by the data's end date)
weekly = so.first_seen.dt.tz_localize(None).dt.to_period("W").value_counts().sort_index()
full = weekly[(weekly.index > FIRST_WEEK) & (weekly.index < LAST_WEEK)]
show("Q2d. Weekly arrivals (full weeks only): enough weeks for a check cadence?", pd.DataFrame({"value": {
    "full weeks": full.size,
    "median new listings per week": compact(full.median()),
    "lowest and highest week, vs. the median": f"{full.min() / full.median():.2f}× and {full.max() / full.median():.2f}×",
    "complete years of monthly data (seasonal decomposition needs 2+)": so.month.nunique() // 12,
}}))

In [ ]:
# Q2e: which listings count as still open? Our rule vs. the hackathon app's (merge decision 2)
# Ours: last seen within 2 days of September 30. The app's signals: last seen within 7 days and not marked closed.
# The app's listing filter today: status not "closed" (status is only ever "closed" or empty).
days_since = (DATA_END - so.last_seen).dt.days
not_closed = so.status.ne("closed").fillna(True).astype(bool)
rules = {
    "ours: last seen within 2 days": so.live.astype(bool),
    "last seen within 7 days": days_since <= 7,
    "the app's signals: within 7 days and not closed": (days_since <= 7) & not_closed,
    "the app's listings today: status not closed": not_closed,
}
show("Q2e. Share of scope listings counted as still open, by rule", pd.DataFrame({
    "share counted open": {k: v.mean() for k, v in rules.items()},
    "share also open under ours (2 days)": {k: (v & rules["ours: last seen within 2 days"]).sum() / max(v.sum(), 1) for k, v in rules.items()},
    "share last seen 30+ days before Sep 30": {k: (v & (days_since > 30)).sum() / max(v.sum(), 1) for k, v in rules.items()},
}))

In [ ]:
# Q2 chart: new listings per week (partial first and last weeks greyed, labelled)
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ax.plot([p.start_time for p in full.index], full.values, color=BLUE, marker="o", ms=4, label="full week")
partial = weekly.drop(full.index)
ax.scatter([p.start_time for p in partial.index], partial.values, color=MUTED, s=40, zorder=3,
           label="partial week (left out of weekly rates)")
ax.set(title="New scope listings per week (first seen)", ylabel="listings")
ax.legend(fontsize=8.5)
finish()

In [ ]:
# Q2 charts: weekday shares and the posted-to-first-seen lag
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
x = np.arange(len(days))
for k, (col, color) in enumerate([("posted (CET date)", BLUE), ("first_seen, all listings", ORANGE)]):
    ax1.bar(x + (k - 0.5) * 0.38, 100 * weekday[col], width=0.36, color=color, label=col)
ax1.set_xticks(x, [d[:3] for d in days])
ax1.set(title="Share of listings by weekday", ylabel="% of listings")
ax1.grid(axis="x", visible=False)
ax1.legend(fontsize=8.5)
ax2.hist(lag.clip(-1, 14), bins=30, color=BLUE)
ax2.set(title="Days from posted to first seen", xlabel="days (capped at 14)", ylabel="listings")
finish()

In [ ]:
# Q2 chart: hour of day of first_seen: the crawler's schedule, not employers'
hours = so.first_seen.dt.hour.value_counts(normalize=True).reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(hours.index, 100 * hours.values, color=MUTED, width=0.7)
ax.set(title="Hour of day listings were first seen (UTC): when the crawler ran", xlabel="hour (UTC)", ylabel="% of listings")
ax.grid(axis="x", visible=False)
finish()

**What the data shows**
- **Six months of steady history:** no seasonal advice, but 75% of scope listings closed by September 30, so timing is well observed. A median **36K new scope listings arrive per full week**; the quietest week is 0.76× the median and the busiest 1.21×.
- **`first_seen` is a good freshness clock:** it lags the employer's posting date by a median 1.5 days (75% within 3.5). Its weekdays follow the crawler (Friday peak), while posting dates (38% of listings) peak Tuesday to Thursday.
- **Still-open rules (merge decision 2):** our rule (last seen within 2 days) counts 25% of scope listings as open, and the app's signal rule (7 days, not closed) 26%, mostly the same listings (92% overlap). **The app's current listing filter (status not closed) counts 42%, and 27% of those were last seen 30+ days before September 30.**

### 9.3 The scope's mix: seniority, description length and other fields

Seniority and description length by role bucket, then the fields the project may use later: language, status values, contract types, what `tags` and `categories` hold, how many listings link to a company in the companies table, and whether the date picker is fair.

In [ ]:
# 9.3a Seniority mix and words per description, by role bucket
levels = {"junior": "junior", "mid_senior": "mid/senior", "manager": "manager"}
seniority = so.seniority.astype(object).map(levels).fillna("director and above / other")
show_shaded("9.3a Seniority mix by role bucket (share of the bucket's listings)",
            pd.crosstab(so.role_bucket, seniority, normalize="index").reindex(BUCKETS))

def words_summary(g):
    w = g.desc_words
    return pd.Series({"median words": w.median(), "25th percentile": w.quantile(0.25),
                      "75th percentile": w.quantile(0.75), "share over 400 words": (w > 400).mean()})
words = so.loc[so.has_desc, ["role_bucket", "desc_words"]].groupby("role_bucket", observed=True).apply(words_summary, include_groups=False)
show_shaded("9.3a Words per description by role bucket (listings with a description)", words.reindex(BUCKETS))

In [ ]:
# 9.3a Chart: seniority mix by role bucket
mix = pd.crosstab(so.role_bucket, seniority, normalize="index").reindex(
    index=BUCKETS, columns=["junior", "mid/senior", "manager", "director and above / other"]).fillna(0)
fig, ax = plt.subplots(figsize=(10, 4.2))
left = np.zeros(len(mix))
for col, color in zip(mix.columns, [BLUE, ORANGE, AQUA, YELLOW]):
    ax.barh(mix.index, 100 * mix[col], left=left, color=color, height=0.65, edgecolor=SURFACE, linewidth=2, label=col)
    left += 100 * mix[col].values
ax.invert_yaxis()
ax.set(title="Seniority mix by role bucket", xlabel="% of the bucket's listings")
ax.grid(axis="y", visible=False)
ax.legend(ncol=4, fontsize=8.5, loc="lower center", bbox_to_anchor=(0.5, -0.32))
finish()

In [ ]:
# 9.3b Other fields: language, status, contract types, tags and categories, company match
def counts(s, n=TOP):
    vc = s.value_counts(normalize=True)
    return vc[vc > 0].head(n).round(3).to_frame("share")

def label_counts(bits, names, n=TOP):
    return (bit_counts(bits, names) / len(so)).head(n).round(3).to_frame("share")

show("Language", counts(with_missing(so.language), 10))
show("Status values", counts(with_missing(so.status)))
contract = label_counts(so.contract_bits, labels["contract_types"])
contract.loc["(none)"] = round((so.contract_bits == 0).mean(), 3)
show("contract_types (a listing can have several)", contract)
show("tags and categories (structure only; distinct values counted over every cleaned listing)", pd.DataFrame({
    "avg items per listing": {"tags": round(so.n_tags.mean(), 1), "categories": round(so.n_categories.mean(), 1)},
    "empty share": {"tags": round((so.n_tags == 0).mean(), 3), "categories": round((so.n_categories == 0).mean(), 3)},
    "distinct values": {"tags": labels["distinct_tags"], "categories": len(labels["categories"])},
}))
show("Top categories", label_counts(so.category_bits, labels["categories"]))
show("Company match rate", pd.DataFrame({"value": {
    "scope listings whose company is in the companies table": pct(so[COMPANY].isin(companies["id"]).mean())}}))

In [ ]:
# 9.3b Chart: contract types
fig, ax = plt.subplots(figsize=(8, 4))
ct = bit_counts(so.contract_bits, labels["contract_types"])
ct.loc["(none)"] = (so.contract_bits == 0).sum()
ct = ct.sort_values(ascending=False).head(12)
bar_h(ax, 100 * ct / len(so), "Contract types (after cleaning; a listing can have several)", "% of listings", "{:.1f}%")
finish()

In [ ]:
# 9.3c Date picker check: scope listings live on the 1st of each month (shares and medians)
# Live on a date = first seen on or before it and last seen on or after it. The data starts on 2026-03-30, so listings
# that opened earlier are missing (left truncation): early dates should look thin and young.
rows = {}
for d in pd.date_range("2026-04-01", "2026-09-01", freq="MS", tz="UTC"):
    age = (d - so.first_seen[(so.first_seen <= d) & (so.last_seen >= d)]).dt.days
    rows[d.strftime("%Y-%m-%d")] = {"live listings": len(age), "median days since first seen": age.median(),
                                    "share first seen in the past 30 days": (age <= 30).mean()}
picker = pd.DataFrame(rows).T
picker.insert(1, "live, vs. September 1 (share)", picker["live listings"] / picker["live listings"].iloc[-1])
show("9.3c Listings live on the 1st of each month", picker)
section_done("9. The scope by role bucket")

**What the data shows**
- **Mid/senior dominates every bucket** (75–91%). Junior is highest in IT support (23%) and other computer roles (21%); IT project management is 44% manager level.
- **Descriptions are long everywhere** (median 664–819 words; 79–88% over 400), so EDA_SDA_3's text work needs chunking for small embedding models.
- **Other fields:** contract types are messy (161 labels; full time 53%, hybrid 19%, contract 8.7%, none 30%), and categories are a clean 27-label taxonomy. Every scope company is in the companies table. **The date picker is fair from July:** live listings reach 79% of September's level, and the share first seen in the past 30 days settles near 50%.

## 10. Findings

**Q1, which factors drive prioritization** (inside the scope)

| Factor | Evidence | Verdict |
|:--|:--|:--|
| Company hiring volume (any field, earlier listings only) | Median 10 days at companies with 500+ listings vs. 50 at 1–4; steady across bands and from May on; Gehan–Wilcoxon p < 0.001 (Q1d) | **Strongest:** model it first, with staffing agencies separated (EDA_SDA_2) |
| Still open / apply-by | 29% gone within a week, median 24.1 days; technical sales slowest (38 days), database fastest (21) (Q1b) | **The target:** apply-by estimates by role bucket |
| Source | Medians from 15 days (LinkedIn, Oracle, Taleo) to 71 (Ashby); company sites 21 (Q1c) | **Strong:** model with company volume, which it tracks |
| Recruiters | Median 11.3 days with a named recruiter vs. 24.3 without; 6.5 days for recruiters with 5+ listings; only 1.2% of listings (Q1h) | **Promising but rare:** test against the staffing flag before using it |
| Pay | Disclosed for 45%; disclosure goes with longer windows (79% vs. 64% open at 7 days); $31–$73 an hour by bucket (Q1a, Q1f) | **Useful:** as a value factor, and disclosure as a timing signal |
| Career-change accessibility | Open to career changers from 9.2% (IT project management) to 44% (IT support); open listings pay a median $37.98 (Q1g) | **Core:** hides and raises listings; rules validated in EDA_SDA_3 |
| Employer intent (text) | Long descriptions 81% vs. 67% open at 7 days; remote wording no gap (Q1f); urgency left to EDA_SDA_3 | **Mixed:** test richer NLP in EDA_SDA_3 |
| Reposts | 6.1% of the scope; back a median 7.6 days later; 32% of job-board listings (6.4, Q1e) | **A feature,** read with the source: many are re-listings |
| Crawl artifacts | 2.4% batch first sightings, 1.2% dropouts; timing barely moves without them (6.8, Q1b check) | **Flags, not filters** |
| Transferable fit | Needs a career changer's profile | Profiles and O*NET skills, in ML_Fit |

**Q2, time information:** posting age is supported for every listing (`first_seen` lags the posting date by a median 1.5 days); day of week only through `posted` (38% of scope listings); no seasonality from six months; weekly rates use 25 full weeks. **Still-open rule:** ours (last seen within 2 days) and the app's signal rule (7 days, not closed) agree on most listings; the app's current status filter counts 27% of its "open" listings that were last seen 30+ days before September 30, which supports a last-seen rule for merge decision 2.

**Q3, how tech compares:** computer and mathematical is the best-paid (1st of 23) and most remote (1st) field, 4th in volume, but 22nd on entry-level roles and openness, and last on asking 2 years or fewer. The scope's design follows from it: hide what a newcomer can't get, rank the rest, and point to the open doors (IT support, business and data analysis, Data Analytics roles).

## 11. The scope table

Saved to `data/clean/analysis_table.parquet` for EDA_SDA_2: the scope's listings (reposts flagged), every column above plus the text and list columns added back from the stream's rows. **Descriptions:** the full text of every scope listing is in `data/clean/scope_descriptions.parquet` (8c), joined by `id`. 11d shows how complete the key fields are; 11e lists every column with its role, meaning and coverage.

In [ ]:
# 11c Save the scope table for EDA_SDA_2: the columns above, plus the text and list columns from the stream's rows (id,
# title, title_norm, job_title, desc_hash, contract_types with unified labels, tags, categories and the raw fields kept
# for EDA_SDA_2's feature screening), added part by part. EDA_SDA_2 reads only the columns it needs.
FINAL_PATH = CLEAN_DIR / "analysis_table.parquet"
start = time.time()
save_final(final, FINAL_PATH)
show("The scope table at a glance", pd.DataFrame({"value": {
    "listings": compact(len(final)),
    "companies": compact(final.company_id.nunique()),
    "role buckets": final.role_bucket.nunique(),
    "full months": final.loc[~final.partial_month, "month"].nunique(),
    "late-March listings (partial_month)": f"{final.partial_month.mean():.1%}",
    "still open on September 30": f"{final.live.mean():.1%}",
    "reposts (flagged)": f"{final.is_repost.mean():.1%}",
    "columns saved": len(pq.read_schema(FINAL_PATH).names),
}}))
print(f"saved {FINAL_PATH.name} in {time.time() - start:.0f}s ({FINAL_PATH.stat().st_size / 1e9:.2f} GB); "
      f"descriptions in {DESC_PATH.name}")

In [ ]:
# 11d Coverage: how complete the key fields are (replaces a preview of rows, which can't show 1 million listings)
KEY_FIELDS = {
    "first and last seen": final.first_seen.notna() & final.last_seen.notna(),
    "role bucket": final.role_bucket.ne("(unmapped 15- code)"),
    "source": final.source_type.notna() & final.source_type.ne("(missing)"),
    "description": final.has_desc.fillna(False).astype(bool),
    "seniority": final.seniority.notna() & final.seniority.ne("not_set"),
    "hourly pay": final.salary_hourly_usd.notna(),
    "employer's posting date": final.posted.notna(),
    "years of experience stated": final.years_required.notna(),
    "country": final.country.notna(),
    "named recruiter": final.recruiter_key.notna(),
}
filled = pd.DataFrame(KEY_FIELDS)
show("11d. Share of scope listings with each key field", filled.mean().round(3).to_frame("share filled"))
n_filled = filled.sum(axis=1).value_counts(normalize=True).sort_index(ascending=False)
show(f"11d. Key fields filled per listing (of {len(KEY_FIELDS)})", pd.DataFrame({
    "share of listings": n_filled.round(3), "cumulative share": n_filled.cumsum().round(3)}).rename_axis("fields filled"))

**What the data shows**
- **Timing, role bucket, source and seniority are complete;** description 78%, pay 45%, years stated 43%, employer posting date 39%.
- **43% of listings have 8+ of the 10 key fields and 94% have 5+,** so ranking can count on the core fields for nearly every listing; pay, years and posting date add detail where present.

In [ ]:
# 11e Feature dictionary: every column in the final table, its role and what it means
FEATURES = {
    # column: (role, prioritization factor, description)
    "id": ("identifier", "", "PredictLeads listing id; joins to scope_descriptions.parquet (full descriptions)"),
    "company_id": ("identifier", "", "PredictLeads company id; links to the companies table"),
    "title": ("text", "employer intent", "Listing title as posted"),
    "row": ("identifier", "", "Row position in the stream's rows table (data/flat/all_fields_rows_full/), for joins back to it"),
    "title_norm": ("helper", "", "Lowercased title with whitespace collapsed; used to match reposts"),
    "job_title": ("text", "transferable fit", "PredictLeads' normalized title; where missing, the posted title lowercased and trimmed"),
    "job_title_fallback": ("helper", "", "job_title is the posted title (PredictLeads gave no normalized title)"),
    "has_title": ("helper", "", "The listing has a title"),
    "repost_fp": ("helper", "", "64-bit fingerprint of company, normalized title and description fingerprint; matches reposts (6.4)"),
    "contract_bits": ("helper", "", "contract_types as bit flags (labels unified), for counts in memory"),
    "category_bits": ("helper", "", "categories as bit flags, for counts in memory"),
    "n_tags": ("category", "transferable fit", "Number of tags"),
    "n_categories": ("category", "transferable fit", "Number of categories"),
    "has_recruiter": ("metadata", "", "A recruiter is named on the listing (a flag only: names and contacts are personal data and aren't kept)"),
    "last_processed": ("time", "", "When PredictLeads last processed the listing; likely moves with last_seen, so audited for leakage before any use"),
    "salary_currency": ("factor", "pay", "Currency the pay was posted in"),
    "salary_low": ("factor", "pay", "Low end of the pay range as posted, in salary_currency"),
    "salary_high": ("factor", "pay", "High end of the pay range as posted, in salary_currency"),
    "was_translated": ("metadata", "", "PredictLeads translated the title"),
    "first_seen": ("time", "freshness", "When the crawler first found the listing (UTC)"),
    "last_seen": ("time", "still open / apply-by", "When the crawler last saw the listing (UTC)"),
    "posted": ("time", "freshness", "Employer's posting date, where given (about 41%); midnight Central European time"),
    "month": ("time", "", "Month the listing was first seen; the monthly time horizon"),
    "role_bucket": ("category", "role bucket", "Role bucket from the O*NET code (src/career_switch.py): the eight computer and mathematical buckets, other computer roles, the two added TripleTen buckets (technical sales, business and data analysis), or an unmapped 15- code"),
    "company_earlier": ("factor", "competition", "The company's listings in any field first seen before this one (6.7a; same-day listings count together; reposts included)"),
    "full_week": ("helper", "", "First seen in a full week (not the partial first or last week); weekly rates use these only"),
    "recruiter_key": ("factor", "recruiters", "Anonymous key per recruiter (fingerprint of name and company with a per-run salt that's never saved); names and contacts aren't kept"),
    "recruiter_title": ("category", "recruiters", "The recruiter's job title, e.g. technical recruiter"),
    "salary_text": ("text", "pay", "Pay as written in the listing"),
    "pay_from_text": ("helper", "pay", "salary_hourly_usd was parsed from salary_text, since salary_data had no numbers (8d)"),
    "location_text": ("text", "", "Location as written in the listing (e.g. a city or 'Remote - US'); for the app's location filter"),
    "url": ("text", "source", "Link to the listing (the app links to it)"),
    "record_type": ("metadata", "", "PredictLeads record type; the same for every listing (kept for completeness)"),
    "company_record_type": ("metadata", "", "Type of the linked company record; the same for every listing (kept for completeness)"),
    "tripleten_program": ("category", "role bucket", "The TripleTen program that trains for the listing's O*NET code (src/career_switch.py), or 'adjacent tech role' where none does (8c)"),
    "batch_first_seen": ("helper", "freshness", "First seen in a one-day batch holding 60%+ of the company's 5+ listings (6.8): the company was added to the crawl, not a hiring surge"),
    "crawl_dropout": ("helper", "still open / apply-by", "Disappeared on a day holding 60%+ of the company's 5+ disappearances (6.8): the crawler likely lost the site, so days_open may end early"),
    "partial_month": ("scope", "", "First seen on 2026-03-30 or 31, the data's first two days: kept, but left out of per-month counts"),
    "days_open": ("target", "still open / apply-by", "Days from first seen to last seen"),
    "live": ("target", "still open / apply-by", "Last seen within 2 days of September 30, the data's end date: still open then, so we don't know when it closes (censored)"),
    "status": ("metadata", "", "PredictLeads status; only 'closed' or missing, so not used as a live flag"),
    "seniority": ("category", "career-change accessibility", "Seniority level, e.g. junior, mid_senior, manager"),
    "occupation_group": ("category", "occupation group", "O*NET occupation group (PredictLeads field onet_data.family); computer and mathematical, or missing where a 15- code placed the listing"),
    "occupation": ("category", "transferable fit", "O*NET occupation name; finer than the occupation group"),
    "onet_code": ("category", "occupation group", "O*NET-SOC code; maps to a role bucket in the computer and mathematical scope (EDA_SDA_2 3.6)"),
    "categories": ("category", "transferable fit", "PredictLeads category labels (27 values)"),
    "tags": ("category", "transferable fit", "PredictLeads tags; sparse (about half empty)"),
    "contract_types": ("category", "transferable fit", "Contract types, e.g. full time, part time, remote; labels unified in 6.2"),
    "language": ("scope", "", "Listing language; the scope table is English only"),
    "country": ("scope", "", "Country of the first listed location; the final table is United States only"),
    "n_locations": ("category", "", "Number of locations the listing names"),
    "source_type": ("factor", "source", "Hiring platform, job board, or company site / other (from the url domain)"),
    "platform": ("factor", "source", "Named hiring platform or job board; empty for company sites"),
    "salary_low_usd": ("factor", "pay", "Low end of the salary range, USD, in its original time unit"),
    "salary_high_usd": ("factor", "pay", "High end of the salary range, USD, in its original time unit"),
    "salary_unit": ("factor", "pay", "Salary time unit: hour, day, week, month or year"),
    "salary_hourly_usd": ("factor", "pay", "Midpoint pay as an hourly rate (salaried pay ÷ 2,080 hours, full time); blank if not disclosed or implausible"),
    "pay_suspect": ("helper", "pay", "Pay was fixed in 6.5: reversed range swapped, or a clearly wrong unit re-read"),
    "has_desc": ("text", "employer intent", "Has a description"),
    "has_html": ("helper", "", "Description contained HTML before cleaning"),
    "desc_words": ("text", "employer intent", "Words in the cleaned description"),
    "desc_hash": ("helper", "", "Fingerprint of the cleaned description; used to match reposts"),
    "mentions_remote": ("text", "employer intent", "Title or description mentions remote work; blank if there's no description and the title doesn't say remote"),
    "urgency": ("text", "employer intent", "Keyword flag for urgency wording (urgent, immediately, ASAP); blank without a description. Saved for EDA_SDA_3, which tests urgency with NLP; not analyzed here"),
    "years_required": ("text", "career-change accessibility", "Most years of experience the description asks for (lower end of each range, up to 20); blank if none stated"),
    "entry_wording": ("text", "career-change accessibility", "Says entry level, no experience required, will train, paid training or apprenticeship; blank without a description"),
    "changer_wording": ("text", "career-change accessibility", "Mentions career changers, transferable skills or non-traditional backgrounds; blank without a description"),
    "degree_required": ("text", "career-change accessibility", "Says a bachelor's, master's or associate degree is required; blank without a description"),
    "equivalent_experience": ("text", "career-change accessibility", "Accepts equivalent experience in place of a degree; blank without a description"),
    "entry_seniority": ("factor", "career-change accessibility", "Seniority label is entry level (junior)"),
    "low_experience_ask": ("factor", "career-change accessibility", "Asks for 2 years of experience or fewer"),
    "career_open": ("factor", "career-change accessibility", "Any sign of being open to career changers: entry-level seniority, 2 years or fewer, entry-level or training wording, or career-changer wording. Blank when nothing says yes and the text is missing. First pass; vetted in EDA_SDA_4"),
    "is_repost": ("factor", "reposts", "This job was listed before"),
    "times_listed": ("factor", "reposts", "How many times this job has been listed so far (1 = first listing)"),
    "days_since_first_listed": ("factor", "reposts", "Days since the job's original listing"),
    "days_since_repost_last_listed": ("factor", "reposts", "Days since the job's previous listing; 0 for the first listing"),
}
ROLE_ORDER_DICT = ["identifier", "time", "target", "factor", "category", "text", "scope", "metadata", "helper"]

def coverage(col):
    """Share of scope listings with a value: from memory if loaded, else read from the file one column at a time."""
    if col in final.columns:
        return round(100 * final[col].notna().mean(), 1)
    c = pq.read_table(FINAL_PATH, columns=[col]).column(0)
    return round(100 * (1 - c.null_count / max(len(c), 1)), 1)

saved = pq.read_schema(FINAL_PATH).names
dictionary = pd.DataFrame(
    [(col, *FEATURES.get(col, ("?", "", "no description yet")), coverage(col)) for col in saved],
    columns=["column", "role", "prioritization factor", "description", "coverage %"],
).set_index("column")
dictionary["low coverage"] = np.where(dictionary["coverage %"] < 20, "under 20%", "")
dictionary = dictionary.sort_values("role", key=lambda s: s.map({r: i for i, r in enumerate(ROLE_ORDER_DICT)}).fillna(99),
                                    kind="stable")
show("Scope table: feature dictionary, by role (coverage = share of scope listings with a value)", dictionary)
undocumented = [c for c in saved if c not in FEATURES]
if undocumented:
    print("columns without a description:", undocumented)
section_done("11. Scope table saved")

## 12. Simple SDA: summary statistics of the final table

A short statistical summary of the final table: its size, how its key measures are distributed, and how it splits by month, role bucket and source. Aggregates only.

In [ ]:
# 12a Key measures: distributions
show("Scope table: key measures (description words among listings with a description)", final[["days_open", "salary_hourly_usd"]]
     .assign(desc_words=final.desc_words.where(final.has_desc.fillna(False).astype(bool)))
     .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T)

In [ ]:
# 12b Key flags: share of listings
show("Final table: share of listings with each flag (text flags: among listings with a description)", pd.DataFrame({"share": {
    "closed by September 30": (~final.live).mean(),
    "still open on September 30": final.live.mean(),
    "has a description": final.has_desc.mean(),
    "salary disclosed": final.salary_hourly_usd.notna().mean(),
    "mentions remote": final.mentions_remote.mean(),
    "repost (listed before)": final.is_repost.mean(),
    "entry-level seniority": final.entry_seniority.mean(),
    "asks 2 years or fewer": final.low_experience_ask.mean(),
    "entry-level or training wording": final.entry_wording.mean(),
    "career-changer wording": final.changer_wording.mean(),
    "degree required": final.degree_required.mean(),
    "open to career changers (any sign)": final.career_open.mean(),
}}).round(3))

In [ ]:
# 12c How the scope table splits: by month, role bucket and source (shares)
by_month = final.loc[~final.partial_month, "month"].value_counts().sort_index()  # full months only
show("Scope table: listings by month (full months; late March is flagged partial_month)",
     pd.DataFrame({"share": by_month / by_month.sum()}))
buckets = final.role_bucket.value_counts(normalize=True)
show("Scope table: role buckets", pd.DataFrame({"share": buckets[buckets > 0]}))
show("Scope table: source type", pd.DataFrame({"share": final.source_type.value_counts(normalize=True)}))

In [ ]:
# 12d Chart: listings by month in the scope table
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar([str(m) for m in by_month.index], 100 * by_month.values / by_month.sum(), color=BLUE, width=0.6)
ax.set(title="Scope table: share of listings by month first seen", ylabel="% of listings")
ax.grid(axis="x", visible=False)
finish()

**What the data shows**
- **The scope table: 1.0M listings from 84K companies,** 24% still open on September 30, 6.1% reposts, and an even spread across months (15–18% each).
- **Median 17 days open** (10% over 84), **median pay $63.90 an hour** (middle half $43.70–$84.40), and a median 747 words per description.
- **Career-change signals are thin:** 10% entry level, 7.7% ask for 2 years or fewer, 4.2% use entry-level or training wording and 0.5% invite career changers; 22% show any sign.

## 13. Final conclusion

### Summary

| | |
|:--|:--|
| **Data** | 13,415,388 listings in the file; 13,017,464 English (or unlabeled) and US (or no country); first seen March 30 to September 30, 2026 (the data's end date) |
| **After cleaning** | 13.0M listings, none removed; 1.4M reposts flagged (10%); 11.7M originals from 324K companies; 22.3% still open on September 30 |
| **Scope table** | 1.0M listings (947K originals) from 84K companies in 11 role buckets; 24% still open on September 30; 59% under a TripleTen program, 41% adjacent tech roles |

**Key findings:**
1. **Tech is worth it and hard to enter:** computer and mathematical is the best-paid field ($66.73 an hour vs. $26.44 for the median field) and the most remote (33% vs. 9.5%), but 22nd of 23 on entry-level roles and openness.
2. **Company volume is the strongest timing signal:** listings at companies with 500+ listings in any field close in a median 10 days, vs. 50 at companies with 1–4.
3. **The window is short:** 29% of scope listings are gone within a week (median 24 days); technical sales is the slowest bucket (38 days).
4. **The open doors:** IT support (44% show openness, 23% entry level) and Data Analytics roles (30%); AI Software Engineering and AI & Machine Learning roles pay the most (about $73.50 an hour) but are among the least open (18–19%).
5. **Still-open rules differ in practice:** a status-based filter counts 42% of scope listings as open, 27% of them last seen a month or more before September 30; a last-seen rule (ours: 25%) avoids that.

### Decisions
- **Order:** clean every listing; compare computer and mathematical roles with every other field (section 7); apply the filters we already know we want, first (section 8); explore the scope by role bucket (section 9).
- **Filters (section 8; the computer and mathematical cut is 8c):** English only (listings with no language label dropped); United States only: listings that name the US (every no-country listing dropped, decided 2026-10-12); the roles TripleTen trains for, plus adjacent tech roles: computer and mathematical by occupation group or a 15- O*NET code, plus sales engineers (technical sales: TripleTen's Forward Deployed Engineering and AI Automation) and management and market research analysts (business and data analysis: TripleTen's Data Analytics) (2026-10-10; IT managers and financial analysts out); credentialed math (actuaries, mathematicians) out; the data's first two days (late March) kept, flagged `partial_month`.
- **Kept from every field before the cut:** the occupation summary (pay and timing per occupation, for the compass's "your pay increase"), company activity (listings in any field, newest listing) and each listing's count of its company's earlier listings in any field.
- **Nothing capped, nothing removed in cleaning but data errors:** reposts become a feature; weekly rates leave out the partial first and last weeks, but the listings stay.

### Limits
- **Six months of history,** so no seasonal advice. `first_seen` follows the crawler, so day-of-week patterns need `posted`.
- **We see listings appear and disappear, not hires.** Openness to career changers is what listings *ask for*, not who gets hired.
- **The text flags are keyword rules** with likely false positives ("urgent care") until EDA_SDA_3 validates them.
- **The six-month window misses older listings:** listings first seen before 2026-03-30 aren't in the data, so the earliest dates undercount what was open; the app's date picker starts at July 1.
- **The field comparison (section 7) is descriptive:** fields differ in company size, source mix and seniority, so like-for-like comparisons happen inside the scope (EDA_SDA_2 and 4). The gaps here aren't significance tests; EDA_SDA_4 section 5 tests them.
- **High-volume posters include staffing agencies,** which can inflate company volume; EDA_SDA_2 (2b) flags them.

**Handoff:** the scope table (`analysis_table.parquet`: English US TripleTen-role and adjacent tech listings, April–September 2026 plus late March flagged `partial_month`, reposts flagged, every candidate factor, role buckets and company history in any field), its descriptions (`scope_descriptions.parquet`), the occupation summary and company activity (every field) go to **EDA_SDA_2**, which cleans further (2b), separates ghost listings, checks non-text factors by role bucket and builds the **compass's 2026 side**.